<a href="https://colab.research.google.com/github/uniqueeee777/SML-stability-driven-energy/blob/main/SML03_%E2%80%94_Bootstrap_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00A — Training-Only Temporal Dependence Audit

In [ ]:

# ================================================================
# NOTEBOOK 03 — CELL 00A (REVISED)
# TRAINING-ONLY TEMPORAL DEPENDENCE AUDIT
#
# RESEARCH:
# Stability-Driven Evaluation of Machine Learning Models
# for Appliance Energy Estimation
#
# METHODOLOGICAL REFERENCES:
# - Kunsch (1989)
# - Politis & Romano (1994)
# - Politis & White (2004)
# - Patton, Politis & White (2009)
#
# OBJECTIVES:
# 1. Load the original UCI dataset from its registered ZIP
# 2. Reconstruct the frozen chronological training partition
# 3. Assess temporal dependence using training data only
# 4. Examine stationarity and distributional variation
# 5. Preserve results for bootstrap block-length assessment
#
# IMPORTANT:
# - No bootstrap block length is selected in this cell
# - No model fitting or resampling
# - No figure generation
# - No validation/test metrics are computed
# ================================================================

import json
import hashlib
import warnings
import zipfile

from pathlib import Path

import numpy as np
import pandas as pd

from statsmodels.tsa.stattools import (
    acf,
    adfuller,
    kpss
)

from IPython.display import display


# ================================================================
# 1. PROJECT CONFIGURATION
# ================================================================

ROOT = Path(
    "/content/drive/MyDrive/Stability_Driven_Energy"
)

RAW_DIR = ROOT / "00_raw_data"

SOURCE_ZIP = (
    RAW_DIR / "appliances+energy+prediction.zip"
)

SOURCE_MEMBER = "energydata_complete.csv"

OUT_DIR = (
    ROOT
    / "04_bootstrap"
    / "diagnostics"
)

EXPECTED_TOTAL_ROWS = 19735

FROZEN_TRAIN_ROWS = 11850

EXPECTED_INTERVAL = pd.Timedelta(
    minutes=10
)

TARGET = "Appliances"

ACF_LAGS = [
    1,
    6,
    36,
    72,
    144,
    288,
    432,
    1008
]

# Existing candidate block lengths.
# These are NOT final bootstrap parameters.

BLOCK_CANDIDATES = [
    144,
    432
]


# ================================================================
# 2. VALIDATION HELPERS
# ================================================================

def require(condition, message):

    if not condition:
        raise RuntimeError(message)


def file_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def array_sha256(values):

    values = np.ascontiguousarray(values)

    return hashlib.sha256(
        values.tobytes()
    ).hexdigest()


# ================================================================
# 3. LOCATE AND VERIFY ORIGINAL ZIP DATASET
# ================================================================

require(
    ROOT.is_dir(),
    f"Project root not found: {ROOT}"
)

require(
    SOURCE_ZIP.is_file(),
    "Original UCI ZIP dataset not found:\n"
    f"{SOURCE_ZIP}\n"
    "Verify the registered Notebook 01 source."
)

require(
    zipfile.is_zipfile(SOURCE_ZIP),
    "The registered source is not a valid ZIP archive."
)

print("=" * 72)
print("NOTEBOOK 03 — ORIGINAL SOURCE VERIFICATION")
print("=" * 72)

print("Source ZIP:", SOURCE_ZIP)


# ================================================================
# 4. READ TRAINING DATA DIRECTLY FROM ZIP
# ================================================================

# Read only the first 11,850 chronological observations.
#
# The validation and test partitions are not loaded
# into the diagnostic dataframe.
#
# The original archive remains unchanged.

with zipfile.ZipFile(
    SOURCE_ZIP,
    mode="r"
) as archive:

    members = [
        name
        for name in archive.namelist()
        if (
            not name.endswith("/")
            and Path(name).name == SOURCE_MEMBER
        )
    ]

    require(
        len(members) == 1,
        "Expected exactly one energydata_complete.csv "
        f"inside the ZIP, found: {members}"
    )

    ARCHIVE_MEMBER = members[0]

    source_info = archive.getinfo(
        ARCHIVE_MEMBER
    )

    require(
        source_info.file_size > 0,
        "CSV member inside ZIP is empty."
    )

    with archive.open(
        ARCHIVE_MEMBER,
        mode="r"
    ) as file:

        train = pd.read_csv(
            file,
            nrows=FROZEN_TRAIN_ROWS
        )


require(
    len(train) == FROZEN_TRAIN_ROWS,
    "Training row count mismatch. "
    f"Expected {FROZEN_TRAIN_ROWS}, "
    f"obtained {len(train)}."
)

EXPECTED_ORIGINAL_COLUMNS = [
    "date",
    "Appliances",
    "lights",
    "T1",
    "RH_1",
    "T2",
    "RH_2",
    "T3",
    "RH_3",
    "T4",
    "RH_4",
    "T5",
    "RH_5",
    "T6",
    "RH_6",
    "T7",
    "RH_7",
    "T8",
    "RH_8",
    "T9",
    "RH_9",
    "T_out",
    "Press_mm_hg",
    "RH_out",
    "Windspeed",
    "Visibility",
    "Tdewpoint",
    "rv1",
    "rv2"
]

require(
    list(train.columns) == EXPECTED_ORIGINAL_COLUMNS,
    "Original dataset column structure does not "
    "match the registered UCI dataset."
)

require(
    train.shape == (11850, 29),
    "Unexpected frozen training data shape."
)

print("\nZIP member:", ARCHIVE_MEMBER)
print("Training shape:", train.shape)
print("CSV source: VERIFIED")


# ================================================================
# 5. VERIFY TRAINING CHRONOLOGY
# ================================================================

train["date"] = pd.to_datetime(
    train["date"],
    errors="raise"
)

require(
    train["date"].notna().all(),
    "Missing timestamps in training data."
)

require(
    train["date"].is_monotonic_increasing,
    "Training observations are not chronological."
)

require(
    train["date"].is_unique,
    "Duplicate timestamps detected."
)

training_start = train["date"].iloc[0]
training_end = train["date"].iloc[-1]

intervals = train["date"].diff().dropna()

irregular_intervals = int(
    (
        intervals != EXPECTED_INTERVAL
    ).sum()
)

require(
    irregular_intervals == 0,
    "Irregular timestamp intervals detected. "
    "Temporal resampling requires further review."
)

print("\nCHRONOLOGICAL TRAINING AUDIT")
print("-" * 72)

print("Training observations:", len(train))
print("Training start:", training_start)
print("Training end:", training_end)
print("Expected interval: 10 minutes")
print("Irregular intervals:", irregular_intervals)
print("Chronological order: PASS")


# ================================================================
# 6. VERIFY NUMERICAL VARIABLES
# ================================================================

excluded_columns = {
    "date",
    "rv1",
    "rv2"
}

numeric_columns = [
    col
    for col in EXPECTED_ORIGINAL_COLUMNS
    if col not in excluded_columns
]

for column in numeric_columns:

    train[column] = pd.to_numeric(
        train[column],
        errors="raise"
    )

require(
    not train[numeric_columns].isna().any().any(),
    "Missing numerical values detected."
)

numeric_array = train[
    numeric_columns
].to_numpy(dtype=np.float64)

require(
    np.isfinite(numeric_array).all(),
    "Nonfinite numerical observations detected."
)

require(
    (train[TARGET] >= 0).all(),
    "The target contains negative values."
)

# Preserve original target in Wh.
# Add log1p as a diagnostic series only.
#
# This does NOT change the frozen B_log
# preprocessing configuration.

diagnostic = train[
    numeric_columns
].copy()

diagnostic["log1p_Appliances"] = np.log1p(
    train[TARGET]
)

print("\nNUMERICAL DATA AUDIT")
print("-" * 72)

print("Original physical variables:", len(numeric_columns))
print("Target:", TARGET)
print("Diagnostic log target: log1p_Appliances")
print("Excluded random columns: rv1, rv2")
print("Missing values: 0")
print("Nonfinite observations: 0")
print("Numerical integrity: PASS")


# ================================================================
# 7. TRAINING-ONLY AUTOCORRELATION
# ================================================================

# ACF is computed on the actual training chronology.
#
# Lag interpretation:
# 1    = 10 minutes
# 6    = 1 hour
# 36   = 6 hours
# 72   = 12 hours
# 144  = 24 hours
# 288  = 48 hours
# 432  = 72 hours
# 1008 = 7 days
#
# ACF is descriptive.
# It does not directly determine optimal block length.

acf_rows = []

MAX_LAG = max(ACF_LAGS)

for variable in diagnostic.columns:

    series = diagnostic[
        variable
    ].to_numpy(dtype=np.float64)

    require(
        np.std(series) > 0,
        f"Constant diagnostic variable: {variable}"
    )

    result = acf(
        series,
        nlags=MAX_LAG,
        fft=True,
        missing="raise"
    )

    record = {
        "variable": variable
    }

    for lag in ACF_LAGS:

        record[f"lag_{lag}"] = float(
            result[lag]
        )

    acf_rows.append(record)


acf_table = pd.DataFrame(
    acf_rows
)

PRIORITY_VARIABLES = [
    "Appliances",
    "log1p_Appliances",
    "lights",
    "T1",
    "T9",
    "RH_1",
    "T_out",
    "RH_out"
]

acf_selected = (
    acf_table
    .set_index("variable")
    .loc[PRIORITY_VARIABLES]
    .reset_index()
)


# ================================================================
# 8. STATIONARITY DIAGNOSTICS
# ================================================================

# ADF H0:
# The series contains a unit root.
#
# KPSS H0:
# The series is level stationary.
#
# Both are diagnostics only.
#
# Rejection/non-rejection must not be interpreted
# as definitive proof of strict stationarity.

STATIONARITY_VARIABLES = [
    "Appliances",
    "log1p_Appliances",
    "T1",
    "T9",
    "T_out",
    "RH_out"
]

stationarity_rows = []

for variable in STATIONARITY_VARIABLES:

    values = diagnostic[
        variable
    ].to_numpy(dtype=np.float64)

    with warnings.catch_warnings(
        record=True
    ) as caught_warnings:

        warnings.simplefilter("always")

        adf_result = adfuller(
            values,
            regression="c",
            autolag="AIC"
        )

        kpss_result = kpss(
            values,
            regression="c",
            nlags="auto"
        )

    warning_messages = [
        str(w.message)
        for w in caught_warnings
    ]

    stationarity_rows.append({
        "variable": variable,

        "adf_statistic": float(
            adf_result[0]
        ),

        "adf_pvalue": float(
            adf_result[1]
        ),

        "adf_used_lags": int(
            adf_result[2]
        ),

        "kpss_statistic": float(
            kpss_result[0]
        ),

        "kpss_pvalue": float(
            kpss_result[1]
        ),

        "kpss_used_lags": int(
            kpss_result[2]
        ),

        "diagnostic_warnings": " | ".join(
            warning_messages
        )
    })


stationarity_table = pd.DataFrame(
    stationarity_rows
)


# ================================================================
# 9. TRAINING DISTRIBUTIONAL VARIATION
# ================================================================

# Three consecutive training segments.
#
# These are descriptive segments only,
# NOT additional CV folds or bootstrap samples.

indices_by_segment = np.array_split(
    np.arange(FROZEN_TRAIN_ROWS),
    3
)

segment_rows = []

for segment_number, indices in enumerate(
    indices_by_segment,
    start=1
):

    subset = train.iloc[indices]

    segment_rows.append({
        "segment": segment_number,

        "start_date": (
            subset["date"].iloc[0].isoformat()
        ),

        "end_date": (
            subset["date"].iloc[-1].isoformat()
        ),

        "n_observations": int(
            len(subset)
        ),

        "appliances_mean_wh": float(
            subset["Appliances"].mean()
        ),

        "appliances_sd_wh": float(
            subset["Appliances"].std(ddof=1)
        ),

        "appliances_median_wh": float(
            subset["Appliances"].median()
        ),

        "appliances_q25_wh": float(
            subset["Appliances"].quantile(0.25)
        ),

        "appliances_q75_wh": float(
            subset["Appliances"].quantile(0.75)
        ),

        "appliances_q95_wh": float(
            subset["Appliances"].quantile(0.95)
        )
    })


segment_table = pd.DataFrame(
    segment_rows
)


# ================================================================
# 10. TRAINING-ONLY DAILY PROFILE
# ================================================================

# The hourly profile is descriptive and is
# computed only from training observations.

hourly_table = (
    train.assign(
        hour=train["date"].dt.hour
    )
    .groupby(
        "hour",
        as_index=False
    )
    .agg(
        mean_appliances_wh=(
            "Appliances",
            "mean"
        ),

        sd_appliances_wh=(
            "Appliances",
            "std"
        ),

        median_appliances_wh=(
            "Appliances",
            "median"
        ),

        n_observations=(
            "Appliances",
            "size"
        )
    )
)


# ================================================================
# 11. BOOTSTRAP BLOCK-LENGTH CANDIDATE REGISTER
# ================================================================

# The candidates come from the previously
# discussed experimental protocol.
#
# They are NOT selected as optimal lengths.

block_candidate_table = pd.DataFrame({
    "block_length_observations": BLOCK_CANDIDATES,

    "duration_hours": [
        length * 10 / 60
        for length in BLOCK_CANDIDATES
    ],

    "duration_days": [
        length / 144
        for length in BLOCK_CANDIDATES
    ],

    "status": [
        "CANDIDATE_NOT_SELECTED"
        for _ in BLOCK_CANDIDATES
    ]
})


# ================================================================
# 12. SAVE DIAGNOSTIC OUTPUTS
# ================================================================

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUTS = {
    "acf_all": (
        OUT_DIR
        / "00a_training_acf_all.csv"
    ),

    "acf_selected": (
        OUT_DIR
        / "00a_training_acf_selected.csv"
    ),

    "stationarity": (
        OUT_DIR
        / "00a_training_stationarity.csv"
    ),

    "segments": (
        OUT_DIR
        / "00a_training_segments.csv"
    ),

    "hourly_profile": (
        OUT_DIR
        / "00a_training_hourly_profile.csv"
    ),

    "block_candidates": (
        OUT_DIR
        / "00a_bootstrap_block_candidates.csv"
    ),

    "manifest": (
        OUT_DIR
        / "00a_training_temporal_audit.json"
    )
}


TABLE_OUTPUTS = {
    "acf_all": acf_table,
    "acf_selected": acf_selected,
    "stationarity": stationarity_table,
    "segments": segment_table,
    "hourly_profile": hourly_table,
    "block_candidates": block_candidate_table
}

for name, dataframe in TABLE_OUTPUTS.items():

    dataframe.to_csv(
        OUTPUTS[name],
        index=False,
        float_format="%.10f"
    )


# ================================================================
# 13. CREATE REPRODUCIBILITY MANIFEST
# ================================================================

# Metadata refers to the actual ZIP source.
# Only the frozen training range is analyzed.
#
# ZIP integrity and training chronology are checked.
# Identity against Notebook 02's frozen training
# artifact must still be cross-verified separately.

manifest = {
    "project": (
        "Stability-Driven Evaluation of "
        "Machine Learning Models for "
        "Appliance Energy Estimation"
    ),

    "notebook": "03_bootstrap_stability_experiments",

    "cell": "00A",

    "audit": (
        "training_only_temporal_dependence"
    ),

    "status": "PASS",

    "dataset": {
        "source_zip": str(
            SOURCE_ZIP
        ),

        "source_zip_sha256": file_sha256(
            SOURCE_ZIP
        ),

        "source_member": (
            ARCHIVE_MEMBER
        ),

        "member_crc32": format(
            source_info.CRC,
            "08x"
        ),

        "expected_total_rows": (
            EXPECTED_TOTAL_ROWS
        ),

        "training_rows": int(
            len(train)
        ),

        "training_start": (
            training_start.isoformat()
        ),

        "training_end": (
            training_end.isoformat()
        ),

        "sampling_interval_minutes": 10,

        "irregular_training_intervals": (
            irregular_intervals
        ),

        "target": TARGET,

        "training_target_sha256": (
            array_sha256(
                train["Appliances"].to_numpy(
                    dtype=np.float64
                )
            )
        )
    },

    "methods": {
        "acf_lags": ACF_LAGS,

        "adf": {
            "regression": "c",
            "autolag": "AIC"
        },

        "kpss": {
            "regression": "c",
            "nlags": "auto"
        },

        "training_segments": 3
    },

    "bootstrap_protocol": {
        "methods": [
            "IID",
            "MBB",
            "Stationary"
        ],

        "candidate_block_lengths": (
            BLOCK_CANDIDATES
        ),

        "block_length_selected": False,

        "bootstrap_generated": False,

        "models_fitted": False
    },

    "verification": {
        "original_zip_found": True,
        "expected_csv_member_found": True,
        "training_rows_verified": True,
        "column_structure_verified": True,
        "training_chronology_verified": True,
        "training_numerical_integrity": True,

        "notebook02_training_identity": (
            "PENDING_CROSS_CHECK"
        )
    },

    "outputs_sha256": {
        name: file_sha256(path)
        for name, path in OUTPUTS.items()
        if name != "manifest"
    }
}


with open(
    OUTPUTS["manifest"],
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False
    )


# ================================================================
# 14. VERIFY SAVED OUTPUTS
# ================================================================

for name, path in OUTPUTS.items():

    require(
        path.is_file(),
        f"Output file missing: {name}"
    )

    require(
        path.stat().st_size > 0,
        f"Output file empty: {name}"
    )


# ================================================================
# 15. FINAL DIAGNOSTIC REPORT
# ================================================================

print("\n" + "=" * 72)
print("NOTEBOOK 03 — CELL 00A FINAL DIAGNOSTIC REPORT")
print("=" * 72)

print("\nSOURCE AND TRAINING")
print("-" * 72)

print("Source ZIP:", SOURCE_ZIP)
print("ZIP member:", ARCHIVE_MEMBER)
print("Training observations:", len(train))
print("Training period:", training_start, "to", training_end)
print("Timestamp interval: 10 minutes")
print("Irregular intervals:", irregular_intervals)

print("\n1. TRAINING-ONLY AUTOCORRELATION")
print("-" * 72)

display(
    acf_selected.round(4)
)

print("\n2. STATIONARITY DIAGNOSTICS")
print("-" * 72)

display(
    stationarity_table[[
        "variable",
        "adf_statistic",
        "adf_pvalue",
        "adf_used_lags",
        "kpss_statistic",
        "kpss_pvalue",
        "kpss_used_lags"
    ]].round(5)
)

print("\n3. TRAINING SEGMENT COMPARISON")
print("-" * 72)

display(
    segment_table.round(3)
)

print("\n4. TRAINING HOURLY PROFILE")
print("-" * 72)

display(
    hourly_table.round(3)
)

print("\n5. BOOTSTRAP BLOCK-LENGTH CANDIDATES")
print("-" * 72)

display(
    block_candidate_table
)

print("\n6. AUDIT OUTPUTS")
print("-" * 72)

for name, path in OUTPUTS.items():
    print(f"{name}: {path}")


print("\n" + "=" * 72)
print("CELL 00A — AUDIT STATUS")
print("=" * 72)

print("Original source ZIP: VERIFIED")
print("Training sample size: VERIFIED")
print("Training chronology: VERIFIED")
print("Numerical integrity: VERIFIED")

print(
    "Notebook 02 training identity: "
    "PENDING CROSS-CHECK"
)

print("Temporal diagnostics: COMPLETED")
print("Block length selected: NO")

print("Bootstrap samples generated: NO")
print("Model fitting performed: NO")
print("Figures generated: NO")
print("Validation/test metrics computed: NO")

print("\nFINAL STATUS: PASS — TEMPORAL DIAGNOSTICS")


NOTEBOOK 03 — ORIGINAL SOURCE VERIFICATION
Source ZIP: /content/drive/MyDrive/Stability_Driven_Energy/00_raw_data/appliances+energy+prediction.zip

ZIP member: energydata_complete.csv
Training shape: (11850, 29)
CSV source: VERIFIED

CHRONOLOGICAL TRAINING AUDIT
------------------------------------------------------------------------
Training observations: 11850
Training start: 2016-01-11 17:00:00
Training end: 2016-04-02 23:50:00
Expected interval: 10 minutes
Irregular intervals: 0
Chronological order: PASS

NUMERICAL DATA AUDIT
------------------------------------------------------------------------
Original physical variables: 26
Target: Appliances
Diagnostic log target: log1p_Appliances
Excluded random columns: rv1, rv2
Missing values: 0
Nonfinite observations: 0
Numerical integrity: PASS

NOTEBOOK 03 — CELL 00A FINAL DIAGNOSTIC REPORT

SOURCE AND TRAINING
------------------------------------------------------------------------
Source ZIP: /content/drive/MyDrive/Stability_Driven_En

,variable,lag_1,lag_6,lag_36,lag_72,lag_144,lag_288,lag_432,lag_1008
0,Appliances,0.7568,0.3100,0.0126,-0.1107,0.2071,0.1872,0.1626,0.1937
1,log1p_Appliances,0.8301,0.4918,0.0269,-0.1511,0.3461,0.3072,0.2822,0.2960
2,lights,0.7984,0.5323,0.0090,0.0608,0.2231,0.1662,0.1308,0.2094
3,T1,0.9991,0.9831,0.7089,0.4650,0.7026,0.5107,0.4237,0.3262
4,T9,0.9995,0.9977,0.9822,0.9658,0.9336,0.8362,0.7401,0.4948
5,RH_1,0.9893,0.9143,0.8206,0.7652,0.7346,0.5928,0.4903,0.3036
6,T_out,0.9991,0.9806,0.7063,0.4986,0.7311,0.5802,0.4200,0.1580
7,RH_out,0.9978,0.9414,0.2996,-0.0634,0.5227,0.4118,0.3776,0.3005



2. STATIONARITY DIAGNOSTICS
------------------------------------------------------------------------


,variable,adf_statistic,adf_pvalue,adf_used_lags,kpss_statistic,kpss_pvalue,kpss_used_lags
0,Appliances,-16.53204,0.00000,12,0.07753,0.10,57
1,log1p_Appliances,-15.75203,0.00000,24,0.19478,0.10,62
2,T1,-9.07291,0.00000,7,2.88434,0.01,68
3,T9,-1.67923,0.44184,15,10.86048,0.01,68
4,T_out,-6.38178,0.00000,40,1.22885,0.01,68
5,RH_out,-11.69231,0.00000,40,1.99625,0.01,67



3. TRAINING SEGMENT COMPARISON
------------------------------------------------------------------------


,segment,start_date,end_date,n_observations,appliances_mean_wh,appliances_sd_wh,appliances_median_wh,appliances_q25_wh,appliances_q75_wh,appliances_q95_wh
0,1,2016-01-11T17:00:00,2016-02-08T03:10:00,3950,97.243,117.609,50.0,40.0,90.0,380.0
1,2,2016-02-08T03:20:00,2016-03-06T13:30:00,3950,98.154,99.167,60.0,50.0,100.0,320.0
2,3,2016-03-06T13:40:00,2016-04-02T23:50:00,3950,96.689,99.644,60.0,50.0,100.0,320.0



4. TRAINING HOURLY PROFILE
------------------------------------------------------------------------


,hour,mean_appliances_wh,sd_appliances_wh,median_appliances_wh,n_observations
0,0,52.154,32.352,50.0,492
1,1,49.756,23.751,50.0,492
2,2,46.484,10.963,50.0,492
3,3,45.427,11.890,50.0,492
4,4,47.053,10.871,50.0,492
5,5,46.037,11.070,50.0,492
6,6,48.984,30.047,50.0,492
7,7,66.057,63.290,50.0,492
8,8,101.179,113.621,60.0,492
9,9,112.622,112.009,70.0,492



5. BOOTSTRAP BLOCK-LENGTH CANDIDATES
------------------------------------------------------------------------


,block_length_observations,duration_hours,duration_days,status
0,144,24.0,1.0,CANDIDATE_NOT_SELECTED
1,432,72.0,3.0,CANDIDATE_NOT_SELECTED



6. AUDIT OUTPUTS
------------------------------------------------------------------------
acf_all: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00a_training_acf_all.csv
acf_selected: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00a_training_acf_selected.csv
stationarity: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00a_training_stationarity.csv
segments: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00a_training_segments.csv
hourly_profile: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00a_training_hourly_profile.csv
block_candidates: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00a_bootstrap_block_candidates.csv
manifest: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00a_training_temporal_audit.json

CELL 00A — AUDIT STATUS
Original source ZIP: VERIFIED
Training sample size: VERIFIED
Training chronolo

# 00B — Frozen Notebook 02 Artifact Discovery

In [ ]:

# ================================================================
# NOTEBOOK 03 — CELL 00B
# NOTEBOOK 02 FROZEN ARTIFACT DISCOVERY
#
# PURPOSE:
# Identify existing training/validation files and
# model specifications created by Notebook 02.
#
# READ ONLY:
# - No raw ZIP access
# - No preprocessing or feature engineering
# - No model fitting
# - No bootstrap
# - No validation/test evaluation
# - No figures
# ================================================================

from pathlib import Path
import json
import hashlib
import pandas as pd
from IPython.display import display

ROOT = Path(
    "/content/drive/MyDrive/Stability_Driven_Energy"
)

assert ROOT.is_dir(), f"Project not found: {ROOT}"

PLAN_ID = "3540b29b8201"
BASELINE_ID = "639561d42efc"

EXPECTED = {
    "train": 11850,
    "validation": 4032,
    "features": 27
}

# ================================================================
# 1. KNOWN REGISTERED MODEL ARTIFACTS
# ================================================================

TUNING_DIR = (
    ROOT / "03_model_development"
    / "grid_search" / PLAN_ID
)

MODEL_FILES = {
    "final_selection_csv": (
        TUNING_DIR / "reports"
        / "final_tuned_model_selection.csv"
    ),

    "final_selection_json": (
        TUNING_DIR / "reports"
        / "final_tuned_model_selection.json"
    ),

    "default_parameters": (
        ROOT / "09_experiment_metadata"
        / "model_specification"
        / "default_estimator_parameters.json"
    ),

    "pipeline_blueprints": (
        ROOT / "09_experiment_metadata"
        / "model_specification"
        / "pipeline_blueprints.csv"
    ),

    "notebook02_final_audit": (
        TUNING_DIR / "reports"
        / "notebook02_final_audit.json"
    )
}

print("=" * 72)
print("A. NOTEBOOK 02 REGISTERED MODEL ARTIFACTS")
print("=" * 72)

model_inventory = []

for name, path in MODEL_FILES.items():

    model_inventory.append({
        "artifact": name,
        "exists": path.is_file(),
        "path": str(path)
    })

display(pd.DataFrame(model_inventory))


# ================================================================
# 2. DISCOVER SAVED DATASET FILES
# ================================================================

# Search stored experiment outputs only.
# The original raw-data directory is excluded.
# Heavy tuning checkpoints are excluded.

EXCLUDED_PARTS = {
    "00_raw_data",
    "baseline_screening",
    "grid_search",
    "checkpoints",
    "figures",
    "invalid_checkpoint_archive",
    "sessions"
}

SEARCH_DIRS = [
    ROOT / "02_preprocessed_data",
    ROOT / "03_model_development",
    ROOT / "09_experiment_metadata"
]

EXTENSIONS = {
    ".csv",
    ".parquet",
    ".npz"
}

data_files = []

for directory in SEARCH_DIRS:

    if not directory.is_dir():
        continue

    for path in directory.rglob("*"):

        if not path.is_file():
            continue

        relative_parts = set(
            path.relative_to(ROOT).parts[:-1]
        )

        if relative_parts & EXCLUDED_PARTS:
            continue

        if path.suffix.lower() not in EXTENSIONS:
            continue

        data_files.append(path)

data_files = sorted(set(data_files))

print("\n" + "=" * 72)
print("B. EXISTING DATA ARTIFACTS")
print("=" * 72)

inventory = pd.DataFrame([
    {
        "filename": p.name,
        "relative_path": str(p.relative_to(ROOT)),
        "size_mb": round(
            p.stat().st_size / (1024 ** 2), 3
        )
    }
    for p in data_files
])

if inventory.empty:
    print("No matching saved data files found.")
else:
    display(inventory)


# ================================================================
# 3. INSPECT DATASET SCHEMAS
# ================================================================

# Read headers only for CSV/Parquet.
# Do not open NPZ, pickle, or fitted models.

dataset_candidates = []

for path in data_files:

    try:
        if path.suffix.lower() == ".csv":

            columns = list(
                pd.read_csv(path, nrows=0).columns
            )

        elif path.suffix.lower() == ".parquet":

            import pyarrow.parquet as pq

            columns = (
                pq.read_schema(path).names
            )

        else:
            continue

    except (
        OSError,
        ValueError,
        ImportError,
        TypeError
    ):
        continue

    feature_columns = [
        c for c in columns
        if c not in {
            "date",
            "timestamp",
            "datetime",
            "Appliances",
            "rv1",
            "rv2"
        }
    ]

    has_target = "Appliances" in columns

    has_cyclical_features = {
        "time_sin",
        "time_cos"
    }.issubset(columns)

    # Report candidate data layouts.
    # Do not assume a source is valid from
    # its filename alone.

    if (
        has_target
        or has_cyclical_features
        or len(feature_columns) == 27
    ):

        dataset_candidates.append({
            "filename": path.name,
            "path": str(path),
            "column_count": len(columns),
            "predictor_count": len(feature_columns),
            "has_target": has_target,
            "has_time_sin_cos": has_cyclical_features,
            "has_date": any(
                c in columns
                for c in [
                    "date",
                    "timestamp",
                    "datetime"
                ]
            )
        })


print("\n" + "=" * 72)
print("C. POTENTIAL FROZEN DATASETS")
print("=" * 72)

candidate_df = pd.DataFrame(
    dataset_candidates
)

if candidate_df.empty:
    print(
        "No combined Feature Set B dataset "
        "identified from saved file schemas."
    )
else:
    display(candidate_df)


# ================================================================
# 4. VERIFY FINAL TUNED MODEL SELECTION
# ================================================================

print("\n" + "=" * 72)
print("D. FINAL TUNED MODEL CONFIGURATIONS")
print("=" * 72)

selection_path = MODEL_FILES[
    "final_selection_csv"
]

if selection_path.is_file():

    selection = pd.read_csv(selection_path)

    required = {
        "algorithm",
        "candidate_id",
        "stage",
        "mean_cv_rmse_wh"
    }

    assert required.issubset(
        selection.columns
    ), "Final selection schema mismatch."

    assert len(selection) == 6, (
        "Expected six final tuned models."
    )

    assert selection["algorithm"].is_unique, (
        "Duplicate tuned model selection."
    )

    display(
        selection[[
            "algorithm",
            "candidate_id",
            "stage",
            "mean_cv_rmse_wh"
        ]].sort_values(
            "mean_cv_rmse_wh"
        ).reset_index(drop=True)
    )

    print(
        "Final tuned configurations: 6 / 6"
    )

else:
    print(
        "Final selection file not found."
    )


# ================================================================
# 5. VERIFY FROZEN AUDIT REGISTRATION
# ================================================================

audit_path = MODEL_FILES[
    "notebook02_final_audit"
]

print("\n" + "=" * 72)
print("E. NOTEBOOK 02 FINAL AUDIT")
print("=" * 72)

if audit_path.is_file():

    with open(
        audit_path,
        encoding="utf-8"
    ) as f:
        audit = json.load(f)

    print("Plan ID:", audit.get("plan_id"))
    print("Status:", audit.get("status"))
    print(
        "Preprocessing:",
        audit.get("preprocessing")
    )

    assert audit.get("status") == "PASS", (
        "Notebook 02 final audit not PASS."
    )

    assert audit.get("plan_id") == PLAN_ID, (
        "Tuning plan mismatch."
    )

    assert audit.get("preprocessing") == "B_log", (
        "Preprocessing mismatch."
    )

else:
    print(
        "Final audit JSON was not found "
        "at the registered path."
    )


# ================================================================
# 6. FINAL DISCOVERY REPORT
# ================================================================

print("\n" + "=" * 72)
print("CELL 00B — DISCOVERY SUMMARY")
print("=" * 72)

print("Project root: VERIFIED")
print("Selected preprocessing: B_log")
print("Expected training rows: 11850")
print("Expected validation rows: 4032")
print("Expected predictors: 27")

print(
    "Existing data files discovered:",
    len(data_files)
)

print(
    "Potential dataset files:",
    len(dataset_candidates)
)

print(
    "Final model selection exists:",
    selection_path.is_file()
)

print(
    "Final Notebook 02 audit exists:",
    audit_path.is_file()
)

print("\nNO RAW DATA LOADED")
print("NO DATA TRANSFORMATION PERFORMED")
print("NO MODEL FITTING PERFORMED")
print("NO BOOTSTRAP GENERATED")
print("NO FIGURES GENERATED")

print(
    "\nSTATUS: DISCOVERY COMPLETED — "
    "SOURCE BINDING NOT YET VERIFIED"
)


A. NOTEBOOK 02 REGISTERED MODEL ARTIFACTS


,artifact,exists,path
0,final_selection_csv,True,/content/drive/MyDrive/Stability_Driven_Energy...
1,final_selection_json,True,/content/drive/MyDrive/Stability_Driven_Energy...
2,default_parameters,True,/content/drive/MyDrive/Stability_Driven_Energy...
3,pipeline_blueprints,True,/content/drive/MyDrive/Stability_Driven_Energy...
4,notebook02_final_audit,True,/content/drive/MyDrive/Stability_Driven_Energy...



B. EXISTING DATA ARTIFACTS


,filename,relative_path,size_mb
0,candidate_feature_set_A.csv,02_preprocessed_data/candidate_feature_set_A.csv,4.758
1,candidate_feature_set_B.csv,02_preprocessed_data/candidate_feature_set_B.csv,5.488
2,A_test.csv,02_preprocessed_data/temporal_partitions/A_tes...,0.928
3,A_train.csv,02_preprocessed_data/temporal_partitions/A_tra...,2.867
4,A_validation.csv,02_preprocessed_data/temporal_partitions/A_val...,0.954
5,B_test.csv,02_preprocessed_data/temporal_partitions/B_tes...,1.069
6,B_train.csv,02_preprocessed_data/temporal_partitions/B_tra...,3.300
7,B_validation.csv,02_preprocessed_data/temporal_partitions/B_val...,1.101
8,cv_fold_indices.npz,09_experiment_metadata/cv_design/cv_fold_indic...,0.062
9,cv_fold_manifest.csv,09_experiment_metadata/cv_design/cv_fold_manif...,0.001



C. POTENTIAL FROZEN DATASETS


,filename,path,column_count,predictor_count,has_target,has_time_sin_cos,has_date
0,candidate_feature_set_A.csv,/content/drive/MyDrive/Stability_Driven_Energy...,27,25,True,False,True
1,candidate_feature_set_B.csv,/content/drive/MyDrive/Stability_Driven_Energy...,29,27,True,True,True
2,A_test.csv,/content/drive/MyDrive/Stability_Driven_Energy...,27,25,True,False,True
3,A_train.csv,/content/drive/MyDrive/Stability_Driven_Energy...,27,25,True,False,True
4,A_validation.csv,/content/drive/MyDrive/Stability_Driven_Energy...,27,25,True,False,True
5,B_test.csv,/content/drive/MyDrive/Stability_Driven_Energy...,29,27,True,True,True
6,B_train.csv,/content/drive/MyDrive/Stability_Driven_Energy...,29,27,True,True,True
7,B_validation.csv,/content/drive/MyDrive/Stability_Driven_Energy...,29,27,True,True,True



D. FINAL TUNED MODEL CONFIGURATIONS


,algorithm,candidate_id,stage,mean_cv_rmse_wh
0,LightGBM,lightgbm_s2_002_5e6f3e8cfc,2,86.621460
1,XGBoost,xgboost_s2_003_022e7d5df1,2,86.962158
2,Random Forest,random_forest_s1_003_8dec891b,1,87.639163
3,Bagging,bagging_s1_003_1c7ab48f,1,87.896565
4,SVR,svr_s1_008_57265079,1,91.773061
5,Decision Tree,decision_tree_s1_004_645f4153,1,95.868786


Final tuned configurations: 6 / 6

E. NOTEBOOK 02 FINAL AUDIT
Plan ID: 3540b29b8201
Status: PASS
Preprocessing: B_log

CELL 00B — DISCOVERY SUMMARY
Project root: VERIFIED
Selected preprocessing: B_log
Expected training rows: 11850
Expected validation rows: 4032
Expected predictors: 27
Existing data files discovered: 14
Potential dataset files: 8
Final model selection exists: True
Final Notebook 02 audit exists: True

NO RAW DATA LOADED
NO DATA TRANSFORMATION PERFORMED
NO MODEL FITTING PERFORMED
NO BOOTSTRAP GENERATED
NO FIGURES GENERATED

STATUS: DISCOVERY COMPLETED — SOURCE BINDING NOT YET VERIFIED


# 00C — Frozen Input Binding & Integrity Audit

In [ ]:

# ================================================================
# NOTEBOOK 03 — CELL 00C
# FROZEN NOTEBOOK 02 INPUT BINDING
#
# Source: Notebook 02 final temporal partitions
# Feature set: B (27 predictors)
# Preprocessing specification: B_log
#
# NO RAW DATA
# NO FEATURE RE-ENGINEERING
# NO MODEL FITTING
# NO BOOTSTRAP
# NO FIGURES
# ================================================================

import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ================================================================
# 1. REGISTERED PATHS
# ================================================================

ROOT = Path(
    "/content/drive/MyDrive/Stability_Driven_Energy"
)

DATA_DIR = (
    ROOT / "02_preprocessed_data"
    / "temporal_partitions"
)

RUN_DIR = (
    ROOT / "03_model_development"
    / "grid_search" / "3540b29b8201"
)

SPEC_DIR = (
    ROOT / "09_experiment_metadata"
    / "model_specification"
)

OUTPUT_DIR = (
    ROOT / "04_bootstrap"
    / "input_registry"
)

PATHS = {
    "train": DATA_DIR / "B_train.csv",
    "validation": DATA_DIR / "B_validation.csv",

    "selection_csv": (
        RUN_DIR / "reports"
        / "final_tuned_model_selection.csv"
    ),

    "selection_json": (
        RUN_DIR / "reports"
        / "final_tuned_model_selection.json"
    ),

    "final_audit": (
        RUN_DIR / "reports"
        / "notebook02_final_audit.json"
    ),

    "default_parameters": (
        SPEC_DIR
        / "default_estimator_parameters.json"
    ),

    "pipeline_blueprints": (
        SPEC_DIR / "pipeline_blueprints.csv"
    )
}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


for name, path in PATHS.items():
    require(
        path.is_file() and path.stat().st_size > 0,
        f"Missing or empty artifact: {name}: {path}"
    )


# ================================================================
# 2. LOAD FROZEN FEATURE SET B
# ================================================================

train_df = pd.read_csv(
    PATHS["train"],
    parse_dates=["date"]
)

validation_df = pd.read_csv(
    PATHS["validation"],
    parse_dates=["date"]
)

EXPECTED_ROWS = {
    "train": 11850,
    "validation": 4032
}

require(
    len(train_df) == EXPECTED_ROWS["train"],
    "Training observation count mismatch."
)

require(
    len(validation_df) == EXPECTED_ROWS["validation"],
    "Validation observation count mismatch."
)

require(
    train_df.shape[1] == 29
    and validation_df.shape[1] == 29,
    "Expected 29 columns in Feature Set B."
)

require(
    train_df.columns.tolist()
    == validation_df.columns.tolist(),
    "Training and validation column order differs."
)


# ================================================================
# 3. VERIFY FROZEN FEATURE STRUCTURE
# ================================================================

TARGET = "Appliances"
DATE_COL = "date"

FEATURE_COLUMNS = [
    col for col in train_df.columns
    if col not in [DATE_COL, TARGET]
]

require(
    len(FEATURE_COLUMNS) == 27,
    "Feature Set B must contain 27 predictors."
)

require(
    {"time_sin", "time_cos"}.issubset(
        FEATURE_COLUMNS
    ),
    "Frozen temporal features are missing."
)

require(
    "rv1" not in FEATURE_COLUMNS
    and "rv2" not in FEATURE_COLUMNS,
    "Excluded random predictors are present."
)

require(
    len(FEATURE_COLUMNS)
    == len(set(FEATURE_COLUMNS)),
    "Duplicate predictor names detected."
)

for name, frame in [
    ("training", train_df),
    ("validation", validation_df)
]:
    require(
        frame[FEATURE_COLUMNS + [TARGET]]
        .apply(pd.api.types.is_numeric_dtype)
        .all(),
        f"Non-numeric predictors in {name}."
    )

    require(
        np.isfinite(
            frame[FEATURE_COLUMNS + [TARGET]]
            .to_numpy(dtype=np.float64)
        ).all(),
        f"Missing or nonfinite values in {name}."
    )

    require(
        (frame[TARGET] >= 0).all(),
        f"Negative target values in {name}."
    )

    cyclic_identity = (
        frame["time_sin"].to_numpy() ** 2
        + frame["time_cos"].to_numpy() ** 2
    )

    require(
        np.allclose(
            cyclic_identity,
            1.0,
            atol=1e-5,
            rtol=0
        ),
        f"Invalid cyclic features in {name}."
    )


# ================================================================
# 4. VERIFY FROZEN TEMPORAL PARTITIONS
# ================================================================

EXPECTED_INTERVAL = pd.Timedelta(minutes=10)

for name, frame in [
    ("training", train_df),
    ("validation", validation_df)
]:
    require(
        frame[DATE_COL].notna().all(),
        f"Missing timestamps in {name}."
    )

    require(
        frame[DATE_COL].is_monotonic_increasing,
        f"Chronology mismatch in {name}."
    )

    require(
        frame[DATE_COL].is_unique,
        f"Duplicate timestamps in {name}."
    )

    require(
        (
            frame[DATE_COL].diff().dropna()
            == EXPECTED_INTERVAL
        ).all(),
        f"Timestamp continuity failed in {name}."
    )

require(
    train_df[DATE_COL].iloc[-1]
    + EXPECTED_INTERVAL
    == validation_df[DATE_COL].iloc[0],
    "Training/validation boundary mismatch."
)

EXPECTED_DATES = {
    "train_start": "2016-01-11 17:00:00",
    "train_end": "2016-04-02 23:50:00",
    "validation_start": "2016-04-03 00:00:00",
    "validation_end": "2016-04-30 23:50:00"
}

actual_dates = {
    "train_start": train_df[DATE_COL].iloc[0],
    "train_end": train_df[DATE_COL].iloc[-1],
    "validation_start": validation_df[DATE_COL].iloc[0],
    "validation_end": validation_df[DATE_COL].iloc[-1]
}

for name, expected in EXPECTED_DATES.items():
    require(
        actual_dates[name] == pd.Timestamp(expected),
        f"Frozen date boundary mismatch: {name}"
    )


# ================================================================
# 5. BIND FEATURE MATRICES AND TARGETS
# ================================================================

# Target remains in its stored scale (Wh).
# log1p will be applied inside the registered
# model fitting workflow, not here.
#
# No scaling is fitted in this cell.

X_train = train_df[FEATURE_COLUMNS].copy()

y_train_wh = train_df[TARGET].copy()

X_validation = validation_df[
    FEATURE_COLUMNS
].copy()

y_validation_wh = validation_df[
    TARGET
].copy()

train_timestamps = train_df[
    DATE_COL
].copy()

validation_timestamps = validation_df[
    DATE_COL
].copy()


# ================================================================
# 6. VERIFY FROZEN MODEL REGISTRATION
# ================================================================

with open(
    PATHS["final_audit"],
    encoding="utf-8"
) as f:
    final_audit = json.load(f)

with open(
    PATHS["selection_json"],
    encoding="utf-8"
) as f:
    selection_registration = json.load(f)

with open(
    PATHS["default_parameters"],
    encoding="utf-8"
) as f:
    default_parameters = json.load(f)

pipeline_blueprints = pd.read_csv(
    PATHS["pipeline_blueprints"]
)

selection = pd.read_csv(
    PATHS["selection_csv"]
)

require(
    final_audit.get("status") == "PASS",
    "Notebook 02 final audit is not PASS."
)

require(
    final_audit.get("plan_id") == "3540b29b8201",
    "Notebook 02 tuning plan mismatch."
)

require(
    final_audit.get("preprocessing") == "B_log",
    "Unexpected selected preprocessing."
)

require(
    selection_registration.get("plan_id")
    == "3540b29b8201",
    "Final selection registration mismatch."
)

require(
    selection_registration.get("preprocessing")
    == "B_log",
    "Final selection preprocessing mismatch."
)

expected_selection_hash = (
    selection_registration["output_sha256"][
        "selection_csv"
    ]
)

require(
    sha256_file(PATHS["selection_csv"])
    == expected_selection_hash,
    "Frozen model selection checksum mismatch."
)

if "selection" in final_audit.get(
    "artifact_sha256", {}
):
    require(
        sha256_file(PATHS["selection_csv"])
        == final_audit["artifact_sha256"]["selection"],
        "Independent selection audit mismatch."
    )

require(
    isinstance(default_parameters, dict)
    and len(default_parameters) > 0,
    "Default model specification is empty."
)

require(
    not pipeline_blueprints.empty,
    "Pipeline specification is empty."
)

EXPECTED_MODELS = {
    "Decision Tree",
    "Random Forest",
    "Bagging",
    "SVR",
    "XGBoost",
    "LightGBM"
}

require(
    len(selection) == 6
    and selection["algorithm"].is_unique
    and set(selection["algorithm"]) == EXPECTED_MODELS,
    "Final tuned model registration mismatch."
)


# ================================================================
# 7. CREATE FROZEN INPUT REGISTRY
# ================================================================

# Hashes identify the exact current files.
# The model selection hash is additionally
# checked against the Notebook 02 registration.
#
# No dataset or fitted model is copied.

source_hashes = {
    name: sha256_file(path)
    for name, path in PATHS.items()
}

manifest = {
    "notebook": "03_bootstrap_stability_experiments",
    "cell": "00C",
    "status": "PASS",

    "source_notebook": "02_model_development",
    "plan_id": "3540b29b8201",
    "preprocessing": "B_log",

    "training": {
        "path": str(PATHS["train"]),
        "rows": len(X_train),
        "start": str(train_timestamps.iloc[0]),
        "end": str(train_timestamps.iloc[-1])
    },

    "validation": {
        "path": str(PATHS["validation"]),
        "rows": len(X_validation),
        "start": str(validation_timestamps.iloc[0]),
        "end": str(validation_timestamps.iloc[-1])
    },

    "features": {
        "count": len(FEATURE_COLUMNS),
        "ordered_names": FEATURE_COLUMNS
    },

    "target": {
        "name": TARGET,
        "stored_unit": "Wh",
        "fitting_transformation": "log1p",
        "prediction_inverse": "expm1"
    },

    "model_configuration": {
        "algorithms": sorted(EXPECTED_MODELS),
        "conditions": ["default", "tuned"],
        "tuning_plan_id": "3540b29b8201",
        "tuned_candidate_ids": {
            str(row.algorithm): str(row.candidate_id)
            for row in selection.itertuples(index=False)
        }
    },

    "source_sha256": source_hashes,

    "checks": {
        "dataset_structure": "PASS",
        "training_chronology": "PASS",
        "validation_chronology": "PASS",
        "feature_alignment": "PASS",
        "numerical_integrity": "PASS",
        "model_registration": "PASS",
        "tuned_selection_checksum": "PASS"
    },

    "bootstrap": {
        "samples_generated": False,
        "models_fitted": False,
        "block_length_selected": False
    }
}

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_PATH = (
    OUTPUT_DIR
    / "notebook03_frozen_input_binding.json"
)

# Prevent silent replacement if a prior registry
# refers to different source files.

if MANIFEST_PATH.is_file():

    with open(
        MANIFEST_PATH,
        encoding="utf-8"
    ) as f:
        existing_manifest = json.load(f)

    require(
        existing_manifest.get("source_sha256")
        == source_hashes,
        "Existing Notebook 03 input registry refers "
        "to different source file fingerprints."
    )

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ================================================================
# 8. FINAL REPORT
# ================================================================

print("=" * 70)
print("CELL 00C — FROZEN NOTEBOOK 02 INPUT BINDING")
print("=" * 70)

print("\nA. DATA BINDING")

print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Target unit: Wh")
print("Feature set: B")
print("Predictors:", len(FEATURE_COLUMNS))
print("Preprocessing: B_log")

print("\nB. TEMPORAL PARTITIONS")

partition_table = pd.DataFrame([
    {
        "partition": "Training",
        "observations": len(train_df),
        "start": str(train_timestamps.iloc[0]),
        "end": str(train_timestamps.iloc[-1])
    },
    {
        "partition": "Validation",
        "observations": len(validation_df),
        "start": str(validation_timestamps.iloc[0]),
        "end": str(validation_timestamps.iloc[-1])
    }
])

display(partition_table)

print("\nC. FEATURE SPECIFICATION")

display(pd.DataFrame({
    "order": np.arange(
        1, len(FEATURE_COLUMNS) + 1
    ),
    "predictor": FEATURE_COLUMNS
}))

print("\nD. FROZEN TUNED MODELS")

display(
    selection[[
        "algorithm",
        "candidate_id",
        "stage",
        "mean_cv_rmse_wh"
    ]].sort_values(
        "mean_cv_rmse_wh"
    ).reset_index(drop=True)
)

print("\nE. AUDIT")

for check, status in manifest["checks"].items():
    print(f"{check}: {status}")

print("\nInput manifest:")
print(MANIFEST_PATH)

print("\nFINAL STATUS: PASS")
print("NO DATA TRANSFORMATION PERFORMED")
print("NO MODEL FITTING PERFORMED")
print("NO BOOTSTRAP GENERATED")
print("NO FIGURES GENERATED")
print("NO TEST DATA ACCESSED")


CELL 00C — FROZEN NOTEBOOK 02 INPUT BINDING

A. DATA BINDING
Training: (11850, 27)
Validation: (4032, 27)
Target unit: Wh
Feature set: B
Predictors: 27
Preprocessing: B_log

B. TEMPORAL PARTITIONS


,partition,observations,start,end
0,Training,11850,2016-01-11 17:00:00,2016-04-02 23:50:00
1,Validation,4032,2016-04-03 00:00:00,2016-04-30 23:50:00



C. FEATURE SPECIFICATION


,order,predictor
0,1,lights
1,2,T1
2,3,RH_1
3,4,T2
4,5,RH_2
5,6,T3
6,7,RH_3
7,8,T4
8,9,RH_4
9,10,T5



D. FROZEN TUNED MODELS


,algorithm,candidate_id,stage,mean_cv_rmse_wh
0,LightGBM,lightgbm_s2_002_5e6f3e8cfc,2,86.621460
1,XGBoost,xgboost_s2_003_022e7d5df1,2,86.962158
2,Random Forest,random_forest_s1_003_8dec891b,1,87.639163
3,Bagging,bagging_s1_003_1c7ab48f,1,87.896565
4,SVR,svr_s1_008_57265079,1,91.773061
5,Decision Tree,decision_tree_s1_004_645f4153,1,95.868786



E. AUDIT
dataset_structure: PASS
training_chronology: PASS
validation_chronology: PASS
feature_alignment: PASS
numerical_integrity: PASS
model_registration: PASS
tuned_selection_checksum: PASS

Input manifest:
/content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/input_registry/notebook03_frozen_input_binding.json

FINAL STATUS: PASS
NO DATA TRANSFORMATION PERFORMED
NO MODEL FITTING PERFORMED
NO BOOTSTRAP GENERATED
NO FIGURES GENERATED
NO TEST DATA ACCESSED


In [ ]:

# ============================================================
# NOTEBOOK 03 — CELL 00D
# REFERENCE-BASED BOOTSTRAP BLOCK-LENGTH ESTIMATION
#
# References:
# Politis & White (2004)
# Patton, Politis & White (2009) — correction
#
# Uses frozen B_train.csv from Notebook 02.
#
# NO RAW DATA
# NO MODEL FITTING
# NO BOOTSTRAP GENERATION
# NO VALIDATION / TEST EVALUATION
# NO FIGURE GENERATION
# NO FINAL BLOCK-LENGTH SELECTION
# ============================================================

import sys
import json
import hashlib
import subprocess
import importlib.metadata

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. ENVIRONMENT AND REFERENCE IMPLEMENTATION
# ============================================================

# Pin dependency if absent.
# Existing installation is retained and its
# exact version will be recorded.

try:
    from arch.bootstrap import optimal_block_length
except ImportError:
    print("Installing reference implementation: arch==8.0.0")

    subprocess.check_call([
        sys.executable,
        "-m", "pip", "install", "-q",
        "arch==8.0.0"
    ])

    from arch.bootstrap import optimal_block_length

ARCH_VERSION = importlib.metadata.version("arch")


# ============================================================
# 2. FROZEN INPUT PATHS
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/Stability_Driven_Energy"
)

TRAIN_PATH = (
    ROOT / "02_preprocessed_data"
    / "temporal_partitions" / "B_train.csv"
)

REGISTRY_PATH = (
    ROOT / "04_bootstrap" / "input_registry"
    / "notebook03_frozen_input_binding.json"
)

OUT_DIR = (
    ROOT / "04_bootstrap" / "diagnostics"
)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


require(
    TRAIN_PATH.is_file(),
    "Frozen B_train.csv not found."
)

require(
    REGISTRY_PATH.is_file(),
    "Run Cell 00C first."
)

with open(REGISTRY_PATH, encoding="utf-8") as f:
    registry = json.load(f)

require(
    registry["status"] == "PASS",
    "Frozen input registry is not PASS."
)

require(
    registry["preprocessing"] == "B_log",
    "Unexpected preprocessing configuration."
)

require(
    registry["plan_id"] == "3540b29b8201",
    "Frozen tuning plan mismatch."
)

require(
    sha256_file(TRAIN_PATH)
    == registry["source_sha256"]["train"],
    "Frozen training checksum mismatch."
)


# ============================================================
# 3. LOAD TRAINING DATA ONLY
# ============================================================

train = pd.read_csv(
    TRAIN_PATH,
    parse_dates=["date"]
)

require(
    len(train) == 11850,
    "Unexpected training sample size."
)

require(
    train["date"].is_monotonic_increasing
    and train["date"].is_unique,
    "Training chronology mismatch."
)

require(
    (
        train["date"].diff().dropna()
        == pd.Timedelta(minutes=10)
    ).all(),
    "Training timestamp continuity mismatch."
)

FEATURE_COLUMNS = registry[
    "features"
]["ordered_names"]

require(
    len(FEATURE_COLUMNS) == 27,
    "Expected 27 frozen predictors."
)

require(
    [c for c in train.columns
     if c not in ["date", "Appliances"]]
    == FEATURE_COLUMNS,
    "Frozen predictor order mismatch."
)

require(
    (train["Appliances"] >= 0).all(),
    "Target must be nonnegative."
)


# ============================================================
# 4. SPECIFY DIAGNOSTIC SERIES
# ============================================================

# Main diagnostic series:
# log1p(Appliances), corresponding to B_log.
#
# Secondary series:
# Raw target and representative environmental
# predictors with distinct dependence patterns.
#
# No original features or model targets are modified.

SERIES = pd.DataFrame({
    "log1p_Appliances": np.log1p(
        train["Appliances"].to_numpy(dtype=float)
    ),

    "Appliances": train[
        "Appliances"
    ].to_numpy(dtype=float),

    "T1": train[
        "T1"
    ].to_numpy(dtype=float),

    "T9": train[
        "T9"
    ].to_numpy(dtype=float),

    "RH_out": train[
        "RH_out"
    ].to_numpy(dtype=float)
})

require(
    np.isfinite(
        SERIES.to_numpy(dtype=float)
    ).all(),
    "Nonfinite diagnostic values."
)

require(
    (
        SERIES.std(axis=0, ddof=1) > 0
    ).all(),
    "Constant diagnostic series."
)


# ============================================================
# 5. REFERENCE-BASED BLOCK-LENGTH ESTIMATION
# ============================================================

# Politis-White plug-in estimator, corrected 2009.
#
# The implementation operates column by column.
# It estimates block lengths for:
#   - Stationary Bootstrap
#   - Circular Bootstrap
#
# It does NOT directly estimate optimal
# non-circular Moving Block Bootstrap length.

reference_result = optimal_block_length(SERIES)

# Support documented return names across versions.

if {"b_sb", "b_cb"}.issubset(
    reference_result.columns
):
    sb_col = "b_sb"
    cb_col = "b_cb"

elif {"stationary", "circular"}.issubset(
    reference_result.columns
):
    sb_col = "stationary"
    cb_col = "circular"

else:
    raise RuntimeError(
        "Unrecognized arch block-length output: "
        f"{reference_result.columns.tolist()}"
    )

require(
    set(reference_result.index)
    == set(SERIES.columns),
    "Diagnostic variable identity mismatch."
)

reference_result = reference_result.loc[
    SERIES.columns
]

reference_table = pd.DataFrame({
    "variable": SERIES.columns,

    "stationary_reference": (
        reference_result[sb_col].to_numpy(
            dtype=float
        )
    ),

    "circular_reference": (
        reference_result[cb_col].to_numpy(
            dtype=float
        )
    )
})

reference_values = reference_table[[
    "stationary_reference",
    "circular_reference"
]].to_numpy(dtype=float)

require(
    np.isfinite(reference_values).all(),
    "Nonfinite reference block-length estimate."
)

require(
    (reference_values > 0).all(),
    "Nonpositive block-length estimate."
)


# ============================================================
# 6. INTERPRET LENGTHS IN TEMPORAL UNITS
# ============================================================

# 10-minute sampling:
# 144 observations = 1 day.

reference_table[
    "stationary_hours"
] = (
    reference_table["stationary_reference"]
    / 6
)

reference_table[
    "circular_hours"
] = (
    reference_table["circular_reference"]
    / 6
)

reference_table[
    "stationary_days"
] = (
    reference_table["stationary_reference"]
    / 144
)

reference_table[
    "circular_days"
] = (
    reference_table["circular_reference"]
    / 144
)


# ============================================================
# 7. COMPARE WITH PRE-REGISTERED CALENDAR CANDIDATES
# ============================================================

# These candidates are not selected here.
# The comparison is diagnostic only.

BLOCK_CANDIDATES = [144, 432]

TARGET_REFERENCE = float(
    reference_table.loc[
        reference_table["variable"]
        == "log1p_Appliances",
        "stationary_reference"
    ].iloc[0]
)

comparison_rows = []

for length in BLOCK_CANDIDATES:

    comparison_rows.append({
        "candidate_length": length,

        "duration_hours": length / 6,

        "duration_days": length / 144,

        "stationary_target_reference": (
            TARGET_REFERENCE
        ),

        "absolute_difference": abs(
            length - TARGET_REFERENCE
        ),

        "status": "NOT_SELECTED"
    })

candidate_comparison = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 8. CHECK ESTIMATOR SCOPE
# ============================================================

# Reference output cannot automatically be
# interpreted as optimal for nonlinear-model RMSE.
#
# Nonstationarity in predictors also prevents
# unconditional theoretical-validity claims.

method_register = pd.DataFrame([
    {
        "method": "IID",
        "reference_estimator": "Not applicable",
        "status": "DEFINED_METHOD"
    },
    {
        "method": "MBB",
        "reference_estimator":
            "Circular estimate is diagnostic only",
        "status": "LENGTH_NOT_SELECTED"
    },
    {
        "method": "Stationary",
        "reference_estimator":
            "Politis-White corrected plug-in",
        "status": "LENGTH_NOT_SELECTED"
    }
])


# ============================================================
# 9. SAVE AUDIT RESULTS
# ============================================================

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REFERENCE_CSV = (
    OUT_DIR
    / "00d_reference_block_lengths.csv"
)

COMPARISON_CSV = (
    OUT_DIR
    / "00d_block_candidate_comparison.csv"
)

METHOD_CSV = (
    OUT_DIR
    / "00d_method_scope_register.csv"
)

MANIFEST_PATH = (
    OUT_DIR
    / "00d_block_length_reference_manifest.json"
)

reference_table.to_csv(
    REFERENCE_CSV,
    index=False,
    float_format="%.10f"
)

candidate_comparison.to_csv(
    COMPARISON_CSV,
    index=False,
    float_format="%.10f"
)

method_register.to_csv(
    METHOD_CSV,
    index=False
)


manifest = {
    "cell": "00D",
    "status": "PASS",

    "source": str(TRAIN_PATH),
    "source_sha256": sha256_file(TRAIN_PATH),

    "source_notebook": "02",
    "preprocessing": "B_log",

    "observations": len(train),
    "sampling_interval_minutes": 10,

    "implementation": {
        "library": "arch",
        "version": ARCH_VERSION,
        "function": "optimal_block_length"
    },

    "references": [
        "Politis and White (2004)",
        "Patton, Politis and White (2009)"
    ],

    "main_diagnostic_series":
        "log1p_Appliances",

    "secondary_diagnostic_series": [
        "Appliances",
        "T1",
        "T9",
        "RH_out"
    ],

    "pre_registered_candidates": (
        BLOCK_CANDIDATES
    ),

    "interpretation": {
        "stationary":
            "Corrected plug-in reference estimate",

        "circular":
            "Circular bootstrap reference; "
            "not automatically optimal for MBB",

        "limitation":
            "Estimator targets a statistical "
            "criterion, not nonlinear-model "
            "RMSE stability directly."
    },

    "block_length_selected": False,
    "bootstrap_generated": False,
    "model_fitting_performed": False,

    "outputs_sha256": {
        "reference": sha256_file(REFERENCE_CSV),
        "comparison": sha256_file(COMPARISON_CSV),
        "method_register": sha256_file(METHOD_CSV)
    }
}

if MANIFEST_PATH.exists():

    with open(
        MANIFEST_PATH,
        encoding="utf-8"
    ) as f:
        previous_manifest = json.load(f)

    require(
        previous_manifest.get("source_sha256")
        == manifest["source_sha256"],
        "Existing diagnostic manifest refers "
        "to a different training dataset."
    )

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 10. FINAL REPORT
# ============================================================

print("=" * 72)
print("CELL 00D — REFERENCE-BASED BLOCK-LENGTH AUDIT")
print("=" * 72)

print("Training source: VERIFIED")
print("Training observations:", len(train))
print("Preprocessing: B_log")
print("Reference library: arch", ARCH_VERSION)

print("\nA. REFERENCE BLOCK-LENGTH ESTIMATES")

display(
    reference_table.round(4)
)

print("\nB. CANDIDATE COMPARISON")

display(
    candidate_comparison.round(4)
)

print("\nC. METHOD SCOPE")

display(method_register)

print("\nD. SAVED OUTPUTS")

for path in [
    REFERENCE_CSV,
    COMPARISON_CSV,
    METHOD_CSV,
    MANIFEST_PATH
]:
    require(
        path.is_file()
        and path.stat().st_size > 0,
        f"Missing output: {path}"
    )
    print(path)

print("\nFINAL STATUS: PASS")
print("Reference estimates calculated: YES")
print("Final block length selected: NO")
print("Bootstrap samples generated: NO")
print("Models fitted: NO")
print("Figures generated: NO")
print("Validation/test evaluated: NO")


Installing reference implementation: arch==8.0.0
CELL 00D — REFERENCE-BASED BLOCK-LENGTH AUDIT
Training source: VERIFIED
Training observations: 11850
Preprocessing: B_log
Reference library: arch 8.0.0

A. REFERENCE BLOCK-LENGTH ESTIMATES


,variable,stationary_reference,circular_reference,stationary_hours,circular_hours,stationary_days,circular_days
0,log1p_Appliances,95.0496,108.8046,15.8416,18.1341,0.6601,0.7556
1,Appliances,90.0464,103.0774,15.0077,17.1796,0.6253,0.7158
2,T1,255.1207,292.0402,42.5201,48.6734,1.7717,2.0281
3,T9,284.0046,325.1041,47.3341,54.1840,1.9723,2.2577
4,RH_out,142.5601,163.1906,23.7600,27.1984,0.9900,1.1333



B. CANDIDATE COMPARISON


,candidate_length,duration_hours,duration_days,stationary_target_reference,absolute_difference,status
0,144,24.0,1.0,95.0496,48.9504,NOT_SELECTED
1,432,72.0,3.0,95.0496,336.9504,NOT_SELECTED



C. METHOD SCOPE


,method,reference_estimator,status
0,IID,Not applicable,DEFINED_METHOD
1,MBB,Circular estimate is diagnostic only,LENGTH_NOT_SELECTED
2,Stationary,Politis-White corrected plug-in,LENGTH_NOT_SELECTED



D. SAVED OUTPUTS
/content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00d_reference_block_lengths.csv
/content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00d_block_candidate_comparison.csv
/content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00d_method_scope_register.csv
/content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/diagnostics/00d_block_length_reference_manifest.json

FINAL STATUS: PASS
Reference estimates calculated: YES
Final block length selected: NO
Bootstrap samples generated: NO
Models fitted: NO
Figures generated: NO
Validation/test evaluated: NO


# Bagian Baru

# 01: Bootstrap Generator & Pilot Indices

In [ ]:

# ================================================================
# NOTEBOOK 03 — CELL 01
# BOOTSTRAP GENERATORS AND INDEX VERIFICATION
#
# Efron (1979)          : IID bootstrap
# Kunsch (1989)         : Moving Block Bootstrap
# Politis & Romano (1994): Stationary Bootstrap
#
# Frozen input: Notebook 02, Feature Set B, B_log
#
# NO RAW DATA
# NO FEATURE ENGINEERING
# NO MODEL FITTING
# NO METRIC EVALUATION
# NO FIGURES
# ================================================================

from pathlib import Path
import hashlib
import json
import os
import math

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------
# 1. FROZEN CONFIGURATION
# ------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Stability_Driven_Energy")

TRAIN_FILE = (
    ROOT / "02_preprocessed_data"
    / "temporal_partitions" / "B_train.csv"
)

VALIDATION_FILE = (
    ROOT / "02_preprocessed_data"
    / "temporal_partitions" / "B_validation.csv"
)

RUN_DIR = (
    ROOT / "03_model_development"
    / "grid_search" / "3540b29b8201"
)

SELECTION_FILE = (
    RUN_DIR / "reports"
    / "final_tuned_model_selection.csv"
)

AUDIT_FILE = (
    RUN_DIR / "reports"
    / "notebook02_final_audit.json"
)

OUTPUT_DIR = (
    ROOT / "04_bootstrap"
    / "resampling_indices"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLAN_ID = "3540b29b8201"
PREPROCESSING = "B_log"

MASTER_SEED = 42
PILOT_B = 30

# Fixed design grid, not data-optimized.
BLOCK_LENGTHS = (72, 144, 432)

EXPERIMENT_VERSION = (
    "bootstrap_fixed_grid_v1"
)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)
    return digest.hexdigest()


def array_hash(array):
    contiguous = np.ascontiguousarray(array)
    return hashlib.sha256(
        contiguous.tobytes()
    ).hexdigest()


# ------------------------------------------------
# 2. LOAD FROZEN NOTEBOOK 02 INPUTS
# ------------------------------------------------

for path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    SELECTION_FILE,
    AUDIT_FILE
]:
    require(
        path.is_file(),
        f"Missing Notebook 02 artifact: {path}"
    )

with open(AUDIT_FILE, encoding="utf-8") as f:
    notebook02_audit = json.load(f)

require(
    notebook02_audit.get("status") == "PASS",
    "Notebook 02 audit must be PASS."
)

require(
    notebook02_audit.get("plan_id") == PLAN_ID,
    "Notebook 02 plan mismatch."
)

require(
    notebook02_audit.get("preprocessing")
    == PREPROCESSING,
    "Notebook 02 preprocessing mismatch."
)

train = pd.read_csv(
    TRAIN_FILE, parse_dates=["date"]
)

validation = pd.read_csv(
    VALIDATION_FILE, parse_dates=["date"]
)

selection = pd.read_csv(SELECTION_FILE)

require(len(train) == 11850, "Training size mismatch.")
require(
    len(validation) == 4032,
    "Validation size mismatch."
)

require(
    train.columns.tolist()
    == validation.columns.tolist(),
    "Feature order differs between partitions."
)

features = [
    c for c in train.columns
    if c not in ("date", "Appliances")
]

require(
    len(features) == 27,
    "Expected 27 predictors."
)

require(
    {"time_sin", "time_cos"}.issubset(features),
    "Frozen temporal features missing."
)

require(
    not {"rv1", "rv2"}.intersection(features),
    "Excluded predictors found."
)

require(
    train["date"].is_monotonic_increasing
    and train["date"].is_unique,
    "Training chronology invalid."
)

require(
    (
        train["date"].diff().dropna()
        == pd.Timedelta(minutes=10)
    ).all(),
    "Training interval mismatch."
)

require(
    train["date"].iloc[-1]
    < validation["date"].iloc[0],
    "Training/validation overlap."
)

require(
    all(
        pd.api.types.is_numeric_dtype(train[c])
        for c in features + ["Appliances"]
    ),
    "Non-numeric frozen data."
)

require(
    np.isfinite(
        train[features + ["Appliances"]]
        .to_numpy(dtype=float)
    ).all(),
    "Nonfinite training values."
)

require(
    (train["Appliances"] >= 0).all(),
    "Negative target values."
)

expected_models = {
    "Decision Tree", "Random Forest", "Bagging",
    "SVR", "XGBoost", "LightGBM"
}

require(
    len(selection) == 6
    and selection["algorithm"].is_unique
    and set(selection["algorithm"]) == expected_models,
    "Tuned model registration mismatch."
)

N = len(train)
TRAIN_SHA256 = sha256_file(TRAIN_FILE)
VALIDATION_SHA256 = sha256_file(VALIDATION_FILE)
SELECTION_SHA256 = sha256_file(SELECTION_FILE)

# Model fitting will use these frozen objects.
# No transformations are applied here.
X_train = train[features].copy()
y_train_wh = train["Appliances"].copy()

X_validation = validation[features].copy()
y_validation_wh = validation["Appliances"].copy()


# ------------------------------------------------
# 3. DEFINE SEVEN RESAMPLING CONDITIONS
# ------------------------------------------------

CONDITIONS = [("iid", None)]

for length in BLOCK_LENGTHS:
    CONDITIONS.append(("mbb", length))

for length in BLOCK_LENGTHS:
    CONDITIONS.append(("stationary", length))


# ------------------------------------------------
# 4. DETERMINISTIC PER-REPLICATION SEEDS
# ------------------------------------------------

def get_seed(method, length, replication):
    key = (
        f"{EXPERIMENT_VERSION}|{PLAN_ID}|"
        f"{TRAIN_SHA256}|{MASTER_SEED}|"
        f"{method}|{length}|{replication}"
    )
    digest = hashlib.sha256(
        key.encode("utf-8")
    ).digest()
    return int.from_bytes(digest[:16], "little")


# ------------------------------------------------
# 5. IID BOOTSTRAP — EFRON (1979)
# ------------------------------------------------

def iid_indices(n, rng):
    indices = rng.integers(
        low=0,
        high=n,
        size=n,
        dtype=np.int32
    )
    return indices, []


# ------------------------------------------------
# 6. MBB — KUNSCH (1989)
# ------------------------------------------------

def mbb_indices(n, block_length, rng):
    ell = int(block_length)

    require(
        1 <= ell <= n,
        "Invalid MBB block length."
    )

    number_of_blocks = math.ceil(n / ell)

    # Original MBB:
    # overlapping, non-circular blocks
    # with valid starts 0,...,n-ell.

    starts = rng.integers(
        low=0,
        high=n - ell + 1,
        size=number_of_blocks
    )

    output = np.empty(n, dtype=np.int32)
    segments = []

    cursor = 0

    for start in starts:
        start = int(start)
        used = min(ell, n - cursor)

        output[cursor:cursor + used] = (
            np.arange(start, start + used)
        )

        segments.append({
            "position": cursor,
            "start": start,
            "used_length": used,
            "raw_length": ell
        })

        cursor += used

    require(cursor == n, "MBB output incomplete.")

    return output, segments


# ------------------------------------------------
# 7. STATIONARY BOOTSTRAP
#    POLITIS & ROMANO (1994)
# ------------------------------------------------

def stationary_indices(n, mean_length, rng):
    ell = float(mean_length)

    require(
        ell >= 1,
        "Invalid expected block length."
    )

    # Geometric distribution:
    # P(L=k) = p(1-p)^(k-1)
    # E(L) = 1/p

    p = 1.0 / ell

    output = np.empty(n, dtype=np.int32)
    segments = []

    cursor = 0

    while cursor < n:
        raw_length = int(rng.geometric(p))
        start = int(rng.integers(0, n))

        used = min(raw_length, n - cursor)

        # Circular continuation:
        # after n-1, continue at index 0.

        positions = (
            start + np.arange(used)
        ) % n

        output[cursor:cursor + used] = positions

        segments.append({
            "position": cursor,
            "start": start,
            "used_length": used,
            "raw_length": raw_length
        })

        cursor += used

    require(
        cursor == n,
        "Stationary bootstrap output incomplete."
    )

    return output, segments


# ------------------------------------------------
# 8. INDEPENDENT STRUCTURAL CHECKS
# ------------------------------------------------

def verify_indices(indices, segments, method, ell):
    require(
        indices.shape == (N,),
        "Incorrect resampled sample size."
    )

    require(
        np.issubdtype(indices.dtype, np.integer),
        "Bootstrap indices must be integers."
    )

    require(
        indices.min() >= 0
        and indices.max() < N,
        "Bootstrap index out of range."
    )

    if method == "iid":
        require(
            len(segments) == 0,
            "IID must not contain block metadata."
        )
        return

    cursor = 0

    for segment in segments:
        start = segment["start"]
        used = segment["used_length"]
        raw = segment["raw_length"]

        require(
            segment["position"] == cursor,
            "Block position mismatch."
        )

        require(
            used > 0 and raw >= used,
            "Invalid block length."
        )

        if method == "mbb":
            require(
                raw == ell,
                "MBB block length is not fixed."
            )

            require(
                0 <= start <= N - ell,
                "MBB block start invalid."
            )

            expected = np.arange(
                start, start + used
            )

        else:
            expected = (
                start + np.arange(used)
            ) % N

        actual = indices[
            cursor:cursor + used
        ]

        require(
            np.array_equal(actual, expected),
            f"{method} block continuity failed."
        )

        cursor += used

    require(
        cursor == N,
        "Resampling segments do not cover n."
    )


# ------------------------------------------------
# 9. FREEZE EXPERIMENT PLAN
# ------------------------------------------------

plan = {
    "experiment_version": EXPERIMENT_VERSION,
    "notebook02_plan_id": PLAN_ID,
    "preprocessing": PREPROCESSING,
    "feature_set": "B",
    "predictor_order": features,
    "target": "Appliances",
    "target_stored_scale": "Wh",
    "fitting_transform": "log1p",
    "inverse_transform": "expm1",
    "master_seed": MASTER_SEED,
    "pilot_repetitions": PILOT_B,
    "planned_cumulative_repetitions": [30, 100, 300],
    "block_lengths": list(BLOCK_LENGTHS),
    "block_length_basis": (
        "fixed temporal sensitivity grid; "
        "no fitted block-length estimator"
    ),
    "methods": {
        "iid": "replacement over individual training rows",
        "mbb": "overlapping, fixed-length, non-circular",
        "stationary": "geometric length, circular wrapping"
    },
    "training_sha256": TRAIN_SHA256,
    "validation_sha256": VALIDATION_SHA256,
    "model_selection_sha256": SELECTION_SHA256,
    "model_selection": dict(zip(
        selection["algorithm"],
        selection["candidate_id"]
    )),
    "model_conditions": ["default", "tuned"],
    "validation_policy": "fixed; no resampling",
    "test_policy": "untouched",
    "index_pairing_policy": (
        "same sampled training indices across "
        "all 12 model conditions within "
        "each method-length-replication"
    )
}

PLAN_FILE = (
    OUTPUT_DIR / "bootstrap_fixed_grid_plan.json"
)

if PLAN_FILE.exists():
    with open(PLAN_FILE, encoding="utf-8") as f:
        previous_plan = json.load(f)

    require(
        previous_plan == plan,
        "An existing bootstrap plan differs. "
        "Do not overwrite an experiment protocol."
    )
else:
    with open(
        PLAN_FILE, "w", encoding="utf-8"
    ) as f:
        json.dump(
            plan, f, indent=2, ensure_ascii=False
        )


# ------------------------------------------------
# 10. GENERATE PILOT INDICES
# ------------------------------------------------

audit_rows = []

for method, ell in CONDITIONS:
    label = (
        "iid" if method == "iid"
        else f"{method}_L{ell:04d}"
    )

    index_matrix = np.empty(
        (PILOT_B, N),
        dtype=np.int32
    )

    block_counts = []
    raw_lengths = []

    for b in range(1, PILOT_B + 1):
        rng = np.random.default_rng(
            get_seed(method, ell, b)
        )

        if method == "iid":
            idx, segments = iid_indices(N, rng)

        elif method == "mbb":
            idx, segments = mbb_indices(
                N, ell, rng
            )

        else:
            idx, segments = stationary_indices(
                N, ell, rng
            )

        verify_indices(
            idx, segments, method, ell
        )

        index_matrix[b - 1] = idx

        block_counts.append(len(segments))
        raw_lengths.extend(
            s["raw_length"] for s in segments
        )

    # File contains indices only.
    # No duplicated training data is saved.

    output_file = (
        OUTPUT_DIR
        / f"{label}_B0001_0030.npz"
    )

    if output_file.exists():
        with np.load(
            output_file, allow_pickle=False
        ) as existing:
            require(
                "indices" in existing.files,
                "Existing checkpoint schema mismatch."
            )
            require(
                np.array_equal(
                    existing["indices"],
                    index_matrix
                ),
                "Existing checkpoint differs "
                "from deterministic index generation."
            )
        action = "VERIFIED_EXISTING"

    else:
        temporary = output_file.with_suffix(
            ".tmp.npz"
        )

        np.savez_compressed(
            temporary,
            indices=index_matrix
        )

        os.replace(temporary, output_file)
        action = "SAVED"

    # Recheck the saved checkpoint.
    with np.load(
        output_file, allow_pickle=False
    ) as saved:
        require(
            np.array_equal(
                saved["indices"],
                index_matrix
            ),
            "Saved checkpoint verification failed."
        )

    audit_rows.append({
        "condition": label,
        "method": method,
        "block_length": (
            np.nan if ell is None else ell
        ),
        "repetitions": PILOT_B,
        "sample_size": N,
        "mean_number_of_blocks": (
            np.nan if method == "iid"
            else float(np.mean(block_counts))
        ),
        "mean_raw_block_length": (
            np.nan if method == "iid"
            else float(np.mean(raw_lengths))
        ),
        "indices_sha256": array_hash(index_matrix),
        "checkpoint_action": action,
        "status": "PASS"
    })


# ------------------------------------------------
# 11. SAVE PILOT AUDIT
# ------------------------------------------------

audit_df = pd.DataFrame(audit_rows)

AUDIT_CSV = (
    OUTPUT_DIR / "pilot_B30_index_audit.csv"
)

audit_df.to_csv(AUDIT_CSV, index=False)

# Keep an independent record of current
# checkpoint file fingerprints.

checkpoint_registry = {}

for method, ell in CONDITIONS:
    label = (
        "iid" if method == "iid"
        else f"{method}_L{ell:04d}"
    )

    path = (
        OUTPUT_DIR
        / f"{label}_B0001_0030.npz"
    )

    checkpoint_registry[label] = {
        "file": path.name,
        "sha256": sha256_file(path)
    }

REGISTRY_FILE = (
    OUTPUT_DIR
    / "pilot_B30_checkpoint_registry.json"
)

registry = {
    "experiment_version": EXPERIMENT_VERSION,
    "training_sha256": TRAIN_SHA256,
    "pilot_B": PILOT_B,
    "checkpoints": checkpoint_registry,
    "status": "PASS"
}

if REGISTRY_FILE.exists():
    with open(
        REGISTRY_FILE, encoding="utf-8"
    ) as f:
        previous_registry = json.load(f)

    require(
        previous_registry == registry,
        "Existing checkpoint registry mismatch."
    )
else:
    with open(
        REGISTRY_FILE, "w", encoding="utf-8"
    ) as f:
        json.dump(registry, f, indent=2)


# ------------------------------------------------
# 12. FINAL REPORT
# ------------------------------------------------

print("=" * 72)
print("CELL 01 — BOOTSTRAP GENERATOR VERIFICATION")
print("=" * 72)

print("\nSOURCE")
print("Notebook 02: VERIFIED")
print("Training rows:", N)
print("Validation rows:", len(validation))
print("Frozen features:", len(features))
print("Preprocessing:", PREPROCESSING)

print("\nMETHOD CONDITIONS")

display(
    audit_df[[
        "condition",
        "method",
        "block_length",
        "repetitions",
        "sample_size",
        "mean_number_of_blocks",
        "mean_raw_block_length",
        "checkpoint_action",
        "status"
    ]].round(3)
)

print("\nCHECKPOINT DIRECTORY")
print(OUTPUT_DIR)

print("\nEXPERIMENT PLAN")
print(PLAN_FILE)

print("\nFINAL STATUS: PASS")
print("Seven resampling conditions verified.")
print("Pilot indices: 30 per condition.")
print("Training data modified: NO")
print("Model fitting performed: NO")
print("Bootstrap model metrics computed: NO")
print("Validation/test evaluated: NO")
print("Figures generated: NO")


CELL 01 — BOOTSTRAP GENERATOR VERIFICATION

SOURCE
Notebook 02: VERIFIED
Training rows: 11850
Validation rows: 4032
Frozen features: 27
Preprocessing: B_log

METHOD CONDITIONS


,condition,method,block_length,repetitions,sample_size,mean_number_of_blocks,mean_raw_block_length,checkpoint_action,status
0,iid,iid,NaN,30,11850,NaN,NaN,SAVED,PASS
1,mbb_L0072,mbb,72.0,30,11850,165.000,72.000,SAVED,PASS
2,mbb_L0144,mbb,144.0,30,11850,83.000,144.000,SAVED,PASS
3,mbb_L0432,mbb,432.0,30,11850,28.000,432.000,SAVED,PASS
4,stationary_L0072,stationary,72.0,30,11850,168.133,70.759,SAVED,PASS
5,stationary_L0144,stationary,144.0,30,11850,84.633,142.267,SAVED,PASS
6,stationary_L0432,stationary,432.0,30,11850,28.700,429.646,SAVED,PASS



CHECKPOINT DIRECTORY
/content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/resampling_indices

EXPERIMENT PLAN
/content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/resampling_indices/bootstrap_fixed_grid_plan.json

FINAL STATUS: PASS
Seven resampling conditions verified.
Pilot indices: 30 per condition.
Training data modified: NO
Model fitting performed: NO
Bootstrap model metrics computed: NO
Validation/test evaluated: NO
Figures generated: NO


# 02 — Pilot Model Refitting

## Model Specification Reload & Preflight
No model fitting
 Read-only audit


Cell ini akan:
- Memuat parameter default dan tuned dari Notebook 02.
- Memeriksa enam model dan pipeline B_log.
- Memeriksa versi library.
- Memverifikasi ulang checkpoint indeks Cell 01.
- Menampilkan spesifikasi lengkap yang akan dipakai Cell 02B.

In [ ]:

# ================================================================
# NOTEBOOK 03 — CELL 02A
# MODEL SPECIFICATION RELOAD & PREFLIGHT
#
# Source of truth: Notebook 02 saved artifacts
#
# NO MODEL FITTING
# NO HYPERPARAMETER TUNING
# NO BOOTSTRAP GENERATION
# NO VALIDATION METRIC COMPUTATION
# NO TEST ACCESS
# ================================================================

import json
import hashlib
import importlib.metadata
import platform
import sys

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display


# ================================================================
# 1. PROJECT PATHS
# ================================================================

ROOT = Path(
    "/content/drive/MyDrive/Stability_Driven_Energy"
)

SPEC_DIR = (
    ROOT / "09_experiment_metadata"
    / "model_specification"
)

TUNING_DIR = (
    ROOT / "03_model_development"
    / "grid_search"
    / "3540b29b8201"
)

BOOTSTRAP_DIR = (
    ROOT / "04_bootstrap"
    / "resampling_indices"
)

OUTPUT_DIR = (
    ROOT / "04_bootstrap"
    / "model_preflight"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 2. REQUIRED NOTEBOOK 02 ARTIFACTS
# ================================================================

ARTIFACTS = {
    "implementation_audit": (
        SPEC_DIR / "model_implementation_audit.csv"
    ),

    "default_parameters": (
        SPEC_DIR / "default_estimator_parameters.json"
    ),

    "pipeline_blueprints": (
        SPEC_DIR / "pipeline_blueprints.csv"
    ),

    "software_environment": (
        SPEC_DIR / "software_environment.json"
    ),

    "implementation_specification": (
        SPEC_DIR / "model_implementation_specification.json"
    ),

    "tuned_selection_csv": (
        TUNING_DIR / "reports"
        / "final_tuned_model_selection.csv"
    ),

    "tuned_selection_json": (
        TUNING_DIR / "reports"
        / "final_tuned_model_selection.json"
    ),

    "bootstrap_plan": (
        BOOTSTRAP_DIR
        / "bootstrap_fixed_grid_plan.json"
    ),

    "bootstrap_registry": (
        BOOTSTRAP_DIR
        / "pilot_B30_checkpoint_registry.json"
    )
}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


for name, path in ARTIFACTS.items():

    require(
        path.is_file()
        and path.stat().st_size > 0,
        f"Missing or empty artifact: {name}\n{path}"
    )


# ================================================================
# 3. LOAD ORIGINAL SPECIFICATIONS
# ================================================================

implementation_audit = pd.read_csv(
    ARTIFACTS["implementation_audit"]
)

default_parameters = json.loads(
    ARTIFACTS["default_parameters"].read_text(
        encoding="utf-8"
    )
)

pipeline_blueprints = pd.read_csv(
    ARTIFACTS["pipeline_blueprints"]
)

software_environment = json.loads(
    ARTIFACTS["software_environment"].read_text(
        encoding="utf-8"
    )
)

implementation_specification = json.loads(
    ARTIFACTS["implementation_specification"].read_text(
        encoding="utf-8"
    )
)

tuned_selection = pd.read_csv(
    ARTIFACTS["tuned_selection_csv"]
)

tuned_selection_json = json.loads(
    ARTIFACTS["tuned_selection_json"].read_text(
        encoding="utf-8"
    )
)

bootstrap_plan = json.loads(
    ARTIFACTS["bootstrap_plan"].read_text(
        encoding="utf-8"
    )
)

bootstrap_registry = json.loads(
    ARTIFACTS["bootstrap_registry"].read_text(
        encoding="utf-8"
    )
)


# ================================================================
# 4. VERIFY EXPERIMENT IDENTITY
# ================================================================

PLAN_ID = "3540b29b8201"

EXPECTED_MODELS = [
    "Decision Tree",
    "Random Forest",
    "Bagging",
    "SVR",
    "XGBoost",
    "LightGBM"
]

require(
    bootstrap_plan["notebook02_plan_id"] == PLAN_ID,
    "Bootstrap plan ID mismatch."
)

require(
    bootstrap_plan["preprocessing"] == "B_log",
    "Preprocessing mismatch."
)

require(
    bootstrap_plan["feature_set"] == "B",
    "Feature set mismatch."
)

require(
    bootstrap_registry["status"] == "PASS",
    "Cell 01 checkpoint registry is not PASS."
)

require(
    bootstrap_registry["pilot_B"] == 30,
    "Pilot replication count mismatch."
)

require(
    bootstrap_plan["training_sha256"]
    == bootstrap_registry["training_sha256"],
    "Training fingerprints differ."
)

require(
    bootstrap_plan["model_selection_sha256"]
    == ARTIFACTS["tuned_selection_csv"].exists()
    * bootstrap_plan["model_selection_sha256"],
    "Invalid selection fingerprint."
)

require(
    set(tuned_selection["algorithm"])
    == set(EXPECTED_MODELS),
    "Tuned model registry mismatch."
)

require(
    len(tuned_selection) == 6
    and tuned_selection["algorithm"].is_unique,
    "Duplicate or missing tuned models."
)


# ================================================================
# 5. VERIFY PIPELINE BLUEPRINTS — B_LOG
# ================================================================

required_blueprint_columns = {
    "algorithm",
    "configuration",
    "n_features",
    "target_treatment",
    "scaler",
    "example_parameter_key"
}

require(
    required_blueprint_columns.issubset(
        pipeline_blueprints.columns
    ),
    "Pipeline blueprint schema mismatch."
)

b_log_blueprints = (
    pipeline_blueprints.loc[
        pipeline_blueprints["configuration"]
        == "B_log"
    ].copy()
)

require(
    len(b_log_blueprints) == 6,
    "Expected six B_log pipeline blueprints."
)

require(
    set(b_log_blueprints["algorithm"])
    == set(EXPECTED_MODELS),
    "B_log algorithm registry mismatch."
)

require(
    (b_log_blueprints["n_features"] == 27).all(),
    "B_log feature count mismatch."
)

require(
    b_log_blueprints["target_treatment"]
    .astype(str).str.lower().eq("log1p").all(),
    "B_log target treatment mismatch."
)


def normalize_scaler(value):
    if pd.isna(value):
        return "none"

    value = str(value).strip().lower()

    if value in {"", "none", "nan"}:
        return "none"

    return value


b_log_blueprints["scaler_normalized"] = (
    b_log_blueprints["scaler"]
    .apply(normalize_scaler)
)

for row in b_log_blueprints.itertuples():

    expected_scaler = (
        "standardscaler"
        if row.algorithm == "SVR"
        else "none"
    )

    require(
        row.scaler_normalized == expected_scaler,
        f"Scaler mismatch: {row.algorithm}"
    )

    require(
        str(row.example_parameter_key).startswith(
            "regressor__model__"
        ),
        f"Unexpected B_log parameter path: "
        f"{row.algorithm}"
    )


# ================================================================
# 6. VERIFY DEFAULT MODEL REGISTRATION
# ================================================================

require(
    len(implementation_audit) == 6,
    "Implementation audit must contain six algorithms."
)

require(
    set(implementation_audit["algorithm"])
    == set(EXPECTED_MODELS),
    "Implementation audit model mismatch."
)

require(
    isinstance(default_parameters, dict)
    and len(default_parameters) > 0,
    "Default parameter specification is empty."
)


# ================================================================
# 7. VERIFY TUNED MODEL REGISTRATION
# ================================================================

required_tuned_columns = {
    "algorithm",
    "stage",
    "candidate_id",
    "mean_cv_rmse_wh"
}

require(
    required_tuned_columns.issubset(
        tuned_selection.columns
    ),
    "Tuned selection schema mismatch."
)

require(
    tuned_selection["candidate_id"].notna().all(),
    "Missing tuned candidate IDs."
)

require(
    tuned_selection["mean_cv_rmse_wh"]
    .notna().all(),
    "Missing tuned CV performance."
)


# ================================================================
# 8. SOFTWARE ENVIRONMENT AUDIT
# ================================================================

PACKAGE_NAMES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "scikit_learn": "scikit-learn",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm"
}

current_versions = {}

for label, package in PACKAGE_NAMES.items():

    try:
        current_versions[label] = (
            importlib.metadata.version(package)
        )
    except importlib.metadata.PackageNotFoundError:
        current_versions[label] = "NOT_INSTALLED"


version_table = pd.DataFrame([
    {
        "component": name,
        "Notebook_02": (
            software_environment.get(name, "SEE_JSON")
            if isinstance(software_environment, dict)
            else "SEE_JSON"
        ),
        "Current_Colab": version
    }
    for name, version in current_versions.items()
])


# ================================================================
# 9. VERIFY ALL SEVEN INDEX CHECKPOINTS
# ================================================================

expected_conditions = [
    "iid",
    "mbb_L0072",
    "mbb_L0144",
    "mbb_L0432",
    "stationary_L0072",
    "stationary_L0144",
    "stationary_L0432"
]

checkpoint_rows = []

for condition in expected_conditions:

    require(
        condition in bootstrap_registry["checkpoints"],
        f"Missing condition: {condition}"
    )

    item = bootstrap_registry[
        "checkpoints"
    ][condition]

    path = BOOTSTRAP_DIR / item["file"]

    require(
        path.is_file(),
        f"Missing checkpoint: {condition}"
    )

    require(
        sha256_file(path) == item["sha256"],
        f"Checkpoint checksum mismatch: {condition}"
    )

    with np.load(
        path,
        allow_pickle=False
    ) as data:

        require(
            "indices" in data.files,
            f"Missing indices: {condition}"
        )

        indices = data["indices"]

        require(
            indices.shape == (30, 11850),
            f"Incorrect index shape: {condition}"
        )

        require(
            np.issubdtype(
                indices.dtype, np.integer
            ),
            f"Invalid index dtype: {condition}"
        )

        require(
            indices.min() >= 0
            and indices.max() < 11850,
            f"Index out of range: {condition}"
        )

    checkpoint_rows.append({
        "condition": condition,
        "replications": 30,
        "sample_size": 11850,
        "sha256_verified": True,
        "status": "PASS"
    })

checkpoint_audit = pd.DataFrame(
    checkpoint_rows
)


# ================================================================
# 10. DISPLAY EXACT MODEL SPECIFICATIONS
# ================================================================

print("=" * 72)
print("CELL 02A — NOTEBOOK 02 MODEL SPECIFICATION AUDIT")
print("=" * 72)

print("\nA. IMPLEMENTATION AUDIT")

display(
    implementation_audit[
        [
            "algorithm",
            "estimator_class",
            "scaling"
        ]
    ]
)

print("\nB. B_LOG PIPELINE BLUEPRINT")

display(
    b_log_blueprints[
        [
            "algorithm",
            "configuration",
            "n_features",
            "target_treatment",
            "scaler",
            "example_parameter_key"
        ]
    ].reset_index(drop=True)
)

print("\nC. EXACT DEFAULT PARAMETERS")

print(
    json.dumps(
        default_parameters,
        indent=2,
        ensure_ascii=False
    )
)

print("\nD. EXACT TUNED SELECTION")

display(
    tuned_selection.sort_values(
        "algorithm"
    ).reset_index(drop=True)
)

print("\nE. TUNED SELECTION JSON")

print(
    json.dumps(
        tuned_selection_json,
        indent=2,
        ensure_ascii=False
    )[:30000]
)

print("\nF. SOFTWARE VERSION COMPARISON")

display(version_table)

print("\nG. NOTEBOOK 02 IMPLEMENTATION SPECIFICATION")

print(
    json.dumps(
        implementation_specification,
        indent=2,
        ensure_ascii=False
    )[:12000]
)

print("\nH. BOOTSTRAP CHECKPOINT AUDIT")

display(checkpoint_audit)


# ================================================================
# 11. SAVE PREFLIGHT REPORT
# ================================================================

report = {
    "cell": "02A",
    "status": "PASS",
    "plan_id": PLAN_ID,
    "preprocessing": "B_log",

    "training_rows": 11850,
    "validation_rows": 4032,
    "predictors": 27,

    "algorithms": EXPECTED_MODELS,

    "model_conditions": [
        "default",
        "tuned"
    ],

    "pipeline_blueprints_verified": True,
    "default_parameter_artifact_verified": True,
    "tuned_selection_verified": True,
    "bootstrap_checkpoints_verified": True,

    "source_sha256": {
        name: sha256_file(path)
        for name, path in ARTIFACTS.items()
    },

    "current_software_versions": current_versions,

    "model_fitting_performed": False,
    "validation_metrics_computed": False,
    "test_accessed": False
}

REPORT_PATH = (
    OUTPUT_DIR
    / "cell02a_model_specification_preflight.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )


# ================================================================
# 12. FINAL STATUS
# ================================================================

print("\n" + "=" * 72)
print("CELL 02A — PREFLIGHT COMPLETED")
print("=" * 72)

print("Notebook 02 artifacts: VERIFIED")
print("B_log pipeline: VERIFIED")
print("Default parameter source: VERIFIED")
print("Tuned model registration: VERIFIED")
print("Bootstrap indices: 7 / 7 VERIFIED")

print("\nPreflight report:")
print(REPORT_PATH)

print("\nNO MODEL FITTING")
print("NO HYPERPARAMETER TUNING")
print("NO VALIDATION EVALUATION")
print("NO FINAL TEST ACCESS")

print("\nREADY FOR CELL 02B ONLY AFTER AUDIT")


CELL 02A — NOTEBOOK 02 MODEL SPECIFICATION AUDIT

A. IMPLEMENTATION AUDIT


,algorithm,estimator_class,scaling
0,Decision Tree,DecisionTreeRegressor,NaN
1,Random Forest,RandomForestRegressor,NaN
2,Bagging,BaggingRegressor,NaN
3,SVR,SVR,StandardScaler
4,XGBoost,XGBRegressor,NaN
5,LightGBM,LGBMRegressor,NaN



B. B_LOG PIPELINE BLUEPRINT


,algorithm,configuration,n_features,target_treatment,scaler,example_parameter_key
0,Decision Tree,B_log,27,log1p,NaN,regressor__model__max_depth
1,Random Forest,B_log,27,log1p,NaN,regressor__model__max_depth
2,Bagging,B_log,27,log1p,NaN,regressor__model__n_estimators
3,SVR,B_log,27,log1p,StandardScaler,regressor__model__C
4,XGBoost,B_log,27,log1p,NaN,regressor__model__max_depth
5,LightGBM,B_log,27,log1p,NaN,regressor__model__max_depth



C. EXACT DEFAULT PARAMETERS
{
  "Decision Tree": {
    "ccp_alpha": 0.0,
    "criterion": "squared_error",
    "max_depth": null,
    "max_features": null,
    "max_leaf_nodes": null,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "monotonic_cst": null,
    "random_state": 42,
    "splitter": "best"
  },
  "Random Forest": {
    "bootstrap": true,
    "ccp_alpha": 0.0,
    "criterion": "squared_error",
    "max_depth": null,
    "max_features": 1.0,
    "max_leaf_nodes": null,
    "max_samples": null,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "monotonic_cst": null,
    "n_estimators": 100,
    "n_jobs": 1,
    "oob_score": false,
    "random_state": 42,
    "verbose": 0,
    "warm_start": false
  },
  "Bagging": {
    "bootstrap": true,
    "bootstrap_features": false,
    "estimator": null,
    "max_features": 1.0,


,algorithm,stage,candidate_id,parameters_json,mean_cv_rmse_wh,min_cv_rmse_wh,rmse_threshold_wh,rmse_above_min_pct,eligible_candidates,mean_fit_seconds,fastest_fit_saving_vs_rmse_min_pct,sd_cv_rmse_wh,mean_cv_mae_wh,mean_cv_r2,baseline_cv_rmse_wh,rmse_improvement_vs_default_pct
0,Bagging,1,bagging_s1_003_1c7ab48f,"{""estimator__max_depth"":12,""estimator__min_sam...",87.896565,87.896565,88.775530,0.000000,7,4.097542,0.000000,19.916941,43.571012,0.174846,91.406051,3.839447
1,Decision Tree,1,decision_tree_s1_004_645f4153,"{""max_depth"":6,""min_samples_leaf"":5,""min_sampl...",95.868786,95.634804,96.591152,0.244662,8,0.141099,21.268995,15.089801,47.923471,-0.021751,153.707012,37.628880
2,LightGBM,2,lightgbm_s2_002_5e6f3e8cfc,"{""colsample_bytree"":0.7,""learning_rate"":0.03,""...",86.621460,86.232262,87.094585,0.451336,9,0.667680,24.054978,21.298576,41.232679,0.203881,87.768487,1.306879
3,Random Forest,1,random_forest_s1_003_8dec891b,"{""max_depth"":12,""max_features"":0.5,""min_sample...",87.639163,87.242610,88.115036,0.454540,10,7.861111,30.332100,20.829021,41.908293,0.183420,89.231982,1.785031
4,SVR,1,svr_s1_008_57265079,"{""C"":0.1,""epsilon"":0.3,""gamma"":""scale""}",91.773061,91.179902,92.091701,0.650537,3,2.883069,6.479103,22.070958,41.570324,0.105288,94.695341,3.085980
5,XGBoost,2,xgboost_s2_003_022e7d5df1,"{""colsample_bytree"":0.7,""learning_rate"":0.03,""...",86.962158,86.353310,87.216843,0.705066,5,1.465872,66.420618,21.206775,41.093163,0.197642,92.568720,6.056648



E. TUNED SELECTION JSON
{
  "plan_id": "3540b29b8201",
  "selection_version": "final_tuned_selection_v1",
  "preprocessing": "B_log",
  "algorithm_order": [
    "Decision Tree",
    "Random Forest",
    "Bagging",
    "SVR",
    "XGBoost",
    "LightGBM"
  ],
  "selection_rule": {
    "rmse_metric": "mean assessment RMSE (Wh) across 4 expanding-window CV folds",
    "relative_tolerance": 0.01,
    "eligibility": "candidate_rmse <= min_algorithm_rmse * 1.01",
    "priority": [
      "lowest mean fit seconds among eligible",
      "lowest mean RMSE if fit time tied",
      "candidate ID alphabetical if both tied"
    ],
    "comparison_scope": "all 139 Stage 1 and Stage 2 candidates, separately within each algorithm"
  },
  "candidate_counts": {
    "stage1": 119,
    "stage2": 20,
    "combined": 139
  },
  "cv_fit_counts": {
    "stage1": 476,
    "stage2": 80,
    "combined": 556
  },
  "selected_models": [
    {
      "algorithm": "Decision Tree",
      "stage": 1,
      "candidate_

,component,Notebook_02,Current_Colab
0,numpy,2.4.6,2.4.6
1,pandas,2.2.3,2.2.3
2,scipy,1.16.3,1.16.3
3,scikit_learn,1.6.1,1.6.1
4,xgboost,3.4.1,3.4.1
5,lightgbm,4.6.0,4.6.0



G. NOTEBOOK 02 IMPLEMENTATION SPECIFICATION
{
  "notebook": "02_Model_Development",
  "cell": "03_Model_Implementation_Specification",
  "n_algorithms": 6,
  "n_preprocessing_candidates": 4,
  "n_unfitted_blueprints": 24,
  "software_versions": {
    "python": "3.13.18",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "numpy": "2.4.6",
    "pandas": "2.2.3",
    "scipy": "1.16.3",
    "scikit_learn": "1.6.1",
    "xgboost": "3.4.1",
    "lightgbm": "4.6.0",
    "matplotlib": "3.10.0",
    "colab_runtime": "Google Colab",
    "audit_time_utc": "2026-11-22T05:36:24.167031+00:00"
  },
  "reproducibility": {
    "random_seed": 42,
    "n_jobs_for_supported_estimators": 1,
    "hardware_policy": "CPU",
    "global_random_state_mutated": false,
    "model_random_state_fixed": true,
    "baseline_hyperparameter_tuning": false,
    "parallel_cv_jobs": 1,
    "bootstrap_seed_strategy": "To be specified before Notebook 03"
  },
  "model_theory": {
    "Decision Tree": {
      "refer

,condition,replications,sample_size,sha256_verified,status
0,iid,30,11850,True,PASS
1,mbb_L0072,30,11850,True,PASS
2,mbb_L0144,30,11850,True,PASS
3,mbb_L0432,30,11850,True,PASS
4,stationary_L0072,30,11850,True,PASS
5,stationary_L0144,30,11850,True,PASS
6,stationary_L0432,30,11850,True,PASS



CELL 02A — PREFLIGHT COMPLETED
Notebook 02 artifacts: VERIFIED
B_log pipeline: VERIFIED
Default parameter source: VERIFIED
Tuned model registration: VERIFIED
Bootstrap indices: 7 / 7 VERIFIED

Preflight report:
/content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_preflight/cell02a_model_specification_preflight.json

NO MODEL FITTING
NO HYPERPARAMETER TUNING
NO VALIDATION EVALUATION
NO FINAL TEST ACCESS

READY FOR CELL 02B ONLY AFTER AUDIT


## 02B — Full B=30 Bootstrap Model Refitting

In [ ]:

# =====================================================================
# NOTEBOOK 03 — CELL 02B
# FULL B=30 BOOTSTRAP MODEL REFITTING
#
# 7 conditions x 30 replications x 6 algorithms x 2 settings
# = 2520 model fits
#
# INPUT:
#   Frozen Notebook 02 B_log data and model specifications
#   Existing Cell 01 bootstrap index checkpoints
#
# OUTPUT:
#   Individual resumable prediction/metric checkpoints
#   Complete B30 metrics table and execution audit
#
# NO NEW BOOTSTRAP INDICES
# NO HYPERPARAMETER TUNING
# NO FEATURE RE-ENGINEERING
# NO FINAL TEST ACCESS
# NO FIGURES
# =====================================================================

import os
import json
import time
import hashlib
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor
)
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from IPython.display import display


# =====================================================================
# 1. FROZEN PATH REGISTRY
# =====================================================================

ROOT = Path(
    "/content/drive/MyDrive/Stability_Driven_Energy"
)

DATA_DIR = (
    ROOT / "02_preprocessed_data"
    / "temporal_partitions"
)

SPEC_DIR = (
    ROOT / "09_experiment_metadata"
    / "model_specification"
)

RUN_DIR = (
    ROOT / "03_model_development"
    / "grid_search"
    / "3540b29b8201"
)

INDEX_DIR = (
    ROOT / "04_bootstrap"
    / "resampling_indices"
)

RESULT_DIR = (
    ROOT / "04_bootstrap"
    / "model_refitting_B30"
)

CHECKPOINT_DIR = RESULT_DIR / "checkpoints"
REPORT_DIR = RESULT_DIR / "reports"

CHECKPOINT_DIR.mkdir(
    parents=True, exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True, exist_ok=True
)

PATHS = {
    "train": DATA_DIR / "B_train.csv",
    "validation": DATA_DIR / "B_validation.csv",
    "defaults": (
        SPEC_DIR / "default_estimator_parameters.json"
    ),
    "blueprints": (
        SPEC_DIR / "pipeline_blueprints.csv"
    ),
    "software": (
        SPEC_DIR / "software_environment.json"
    ),
    "selection": (
        RUN_DIR / "reports"
        / "final_tuned_model_selection.csv"
    ),
    "final_audit": (
        RUN_DIR / "reports"
        / "notebook02_final_audit.json"
    ),
    "plan": (
        INDEX_DIR / "bootstrap_fixed_grid_plan.json"
    ),
    "index_registry": (
        INDEX_DIR
        / "pilot_B30_checkpoint_registry.json"
    )
}

CELL_VERSION = "cell02b_full_B30_v1"
PLAN_ID = "3540b29b8201"
MODEL_SEED = 42
B = 30


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_array(values):
    array = np.ascontiguousarray(values)

    return hashlib.sha256(
        array.tobytes()
    ).hexdigest()


def canonical_json(value):
    return json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        allow_nan=False
    )


def sha256_object(value):
    return hashlib.sha256(
        canonical_json(value).encode("utf-8")
    ).hexdigest()


for name, path in PATHS.items():
    require(
        path.is_file(),
        f"Missing frozen artifact: {name}: {path}"
    )


# =====================================================================
# 2. LOAD NOTEBOOK 02 & CELL 01 REGISTRIES
# =====================================================================

defaults = load_json(PATHS["defaults"])
software = load_json(PATHS["software"])
final_audit = load_json(PATHS["final_audit"])
plan = load_json(PATHS["plan"])
index_registry = load_json(PATHS["index_registry"])

selection = pd.read_csv(PATHS["selection"])
blueprints = pd.read_csv(PATHS["blueprints"])

require(
    final_audit.get("status") == "PASS",
    "Notebook 02 final audit is not PASS."
)

require(
    final_audit.get("plan_id") == PLAN_ID,
    "Notebook 02 plan ID mismatch."
)

require(
    final_audit.get("preprocessing") == "B_log",
    "Notebook 02 preprocessing mismatch."
)

require(
    plan.get("notebook02_plan_id") == PLAN_ID,
    "Bootstrap plan ID mismatch."
)

require(
    plan.get("experiment_version")
    == "bootstrap_fixed_grid_v1",
    "Unexpected bootstrap experiment version."
)

require(
    plan.get("preprocessing") == "B_log"
    and plan.get("feature_set") == "B",
    "Frozen feature/preprocessing mismatch."
)

require(
    plan.get("master_seed") == 42,
    "Bootstrap master seed changed."
)

require(
    plan.get("pilot_repetitions") == B,
    "Registered B30 design mismatch."
)

require(
    plan.get("block_lengths") == [72, 144, 432],
    "Block-length grid changed."
)

require(
    index_registry.get("status") == "PASS"
    and index_registry.get("pilot_B") == B,
    "Cell 01 registry not PASS."
)

require(
    sha256_file(PATHS["train"])
    == plan["training_sha256"],
    "Frozen training fingerprint mismatch."
)

require(
    sha256_file(PATHS["validation"])
    == plan["validation_sha256"],
    "Frozen validation fingerprint mismatch."
)

require(
    sha256_file(PATHS["selection"])
    == plan["model_selection_sha256"],
    "Frozen model selection fingerprint mismatch."
)

require(
    plan["training_sha256"]
    == index_registry["training_sha256"],
    "Index registry training mismatch."
)


# =====================================================================
# 3. VERIFY SOFTWARE ENVIRONMENT
# =====================================================================

PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "scikit_learn": "scikit-learn",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm"
}

current_versions = {
    name: importlib.metadata.version(package)
    for name, package in PACKAGES.items()
}

for name, current in current_versions.items():

    expected = software.get(name)

    require(
        expected is not None,
        f"Missing Notebook 02 package version: {name}"
    )

    require(
        str(current) == str(expected),
        f"Software version mismatch: {name}: "
        f"Notebook02={expected}, current={current}"
    )


# =====================================================================
# 4. LOAD FROZEN TRAINING AND VALIDATION
# =====================================================================

train_df = pd.read_csv(
    PATHS["train"], parse_dates=["date"]
)

validation_df = pd.read_csv(
    PATHS["validation"], parse_dates=["date"]
)

FEATURES = plan["predictor_order"]

require(
    len(train_df) == 11850,
    "Training count mismatch."
)

require(
    len(validation_df) == 4032,
    "Validation count mismatch."
)

require(
    len(FEATURES) == 27
    and len(set(FEATURES)) == 27,
    "Frozen predictor list invalid."
)

require(
    [
        c for c in train_df.columns
        if c not in ["date", "Appliances"]
    ] == FEATURES,
    "Training feature order mismatch."
)

require(
    [
        c for c in validation_df.columns
        if c not in ["date", "Appliances"]
    ] == FEATURES,
    "Validation feature order mismatch."
)

require(
    {"time_sin", "time_cos"}.issubset(FEATURES),
    "Cyclical features missing."
)

require(
    not {"rv1", "rv2"}.intersection(FEATURES),
    "Excluded features unexpectedly present."
)

require(
    train_df["date"].is_monotonic_increasing
    and validation_df["date"].is_monotonic_increasing,
    "Temporal order mismatch."
)

require(
    train_df["date"].iloc[-1]
    < validation_df["date"].iloc[0],
    "Train/validation overlap."
)

X_train = train_df[FEATURES].copy()
y_train_wh = train_df["Appliances"].copy()

X_validation = validation_df[FEATURES].copy()
y_validation_wh = validation_df[
    "Appliances"
].to_numpy(dtype=np.float64)

for name, values in [
    ("X_train", X_train.to_numpy(dtype=float)),
    ("y_train", y_train_wh.to_numpy(dtype=float)),
    ("X_validation", X_validation.to_numpy(dtype=float)),
    ("y_validation", y_validation_wh)
]:
    require(
        np.isfinite(values).all(),
        f"Nonfinite values found: {name}"
    )

require(
    (y_train_wh >= 0).all()
    and (y_validation_wh >= 0).all(),
    "Target must be nonnegative."
)

N = len(X_train)
N_VAL = len(X_validation)


# =====================================================================
# 5. FROZEN MODEL FACTORY
# =====================================================================

MODEL_CLASSES = {
    "Decision Tree": DecisionTreeRegressor,
    "Random Forest": RandomForestRegressor,
    "Bagging": BaggingRegressor,
    "SVR": SVR,
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor
}

MODEL_SLUGS = {
    "Decision Tree": "decision_tree",
    "Random Forest": "random_forest",
    "Bagging": "bagging",
    "SVR": "svr",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm"
}

MODEL_ORDER = list(MODEL_CLASSES)

require(
    set(defaults) == set(MODEL_ORDER),
    "Default model specification mismatch."
)

require(
    len(selection) == 6
    and selection["algorithm"].is_unique
    and set(selection["algorithm"]) == set(MODEL_ORDER),
    "Tuned model registry mismatch."
)

tuned_parameters = {}
candidate_ids = {}

for row in selection.itertuples(index=False):

    algo = row.algorithm

    tuned_parameters[algo] = json.loads(
        row.parameters_json
    )

    candidate_ids[algo] = str(
        row.candidate_id
    )

require(
    set(candidate_ids) == set(MODEL_ORDER),
    "Missing tuned candidate registration."
)

require(
    plan["model_selection"] == candidate_ids,
    "Cell 01 tuned candidate IDs differ."
)

b_log = blueprints[
    blueprints["configuration"] == "B_log"
].copy()

require(
    len(b_log) == 6
    and set(b_log["algorithm"]) == set(MODEL_ORDER),
    "B_log pipeline blueprint mismatch."
)

require(
    (b_log["n_features"] == 27).all()
    and b_log["target_treatment"].eq("log1p").all(),
    "B_log preprocessing contract mismatch."
)

for row in b_log.itertuples(index=False):

    scaler = (
        "none" if pd.isna(row.scaler)
        else str(row.scaler).strip().lower()
    )

    expected = (
        "standardscaler"
        if row.algorithm == "SVR"
        else "none"
    )

    require(
        scaler == expected,
        f"Scaler mismatch for {row.algorithm}."
    )

    require(
        str(row.example_parameter_key).startswith(
            "regressor__model__"
        ),
        "Unexpected nested pipeline key."
    )


def make_frozen_model(algorithm, setting):

    require(
        setting in {"default", "tuned"},
        "Invalid model setting."
    )

    cls = MODEL_CLASSES[algorithm]

    # Use original Notebook 02 constructor parameters.
    constructor_parameters = dict(
        defaults[algorithm]
    )

    estimator = cls(
        **constructor_parameters
    )

    if setting == "tuned":

        parameters = dict(
            tuned_parameters[algorithm]
        )

        if algorithm == "Bagging":

            # Tuned Bagging contains estimator__ keys.
            # The default Bagging estimator is None,
            # so a DecisionTreeRegressor is required
            # before nested parameters can be applied.

            estimator.set_params(
                estimator=DecisionTreeRegressor(
                    random_state=MODEL_SEED
                )
            )

        estimator.set_params(
            **parameters
        )

    # Notebook 02 preprocessing:
    # SVR: StandardScaler + estimator
    # Others: estimator only.

    steps = []

    if algorithm == "SVR":
        steps.append(
            ("scaler", StandardScaler())
        )

    steps.append(
        ("model", estimator)
    )

    pipeline = Pipeline(steps)

    # Target is stored as Appliances in Wh.
    # Transform only during training.
    # Predictions return to Wh automatically.

    model = TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

    return model


# =====================================================================
# 6. FROZEN RESAMPLING CONDITIONS
# =====================================================================

CONDITIONS = [
    "iid",
    "mbb_L0072",
    "mbb_L0144",
    "mbb_L0432",
    "stationary_L0072",
    "stationary_L0144",
    "stationary_L0432"
]

require(
    list(index_registry["checkpoints"]) == CONDITIONS,
    "Registered condition order differs."
)

require(
    len(CONDITIONS) * B * 6 * 2 == 2520,
    "Unexpected experiment fit count."
)


# =====================================================================
# 7. FREEZE REFITTING PROTOCOL
# =====================================================================

protocol = {
    "cell_version": CELL_VERSION,
    "experiment_version": plan["experiment_version"],
    "plan_id": PLAN_ID,
    "training_sha256": plan["training_sha256"],
    "validation_sha256": plan["validation_sha256"],
    "selection_sha256": plan[
        "model_selection_sha256"
    ],
    "defaults_sha256": sha256_file(
        PATHS["defaults"]
    ),
    "blueprints_sha256": sha256_file(
        PATHS["blueprints"]
    ),
    "conditions": CONDITIONS,
    "replications": B,
    "algorithm_order": MODEL_ORDER,
    "model_settings": ["default", "tuned"],
    "candidate_ids": candidate_ids,
    "model_seed": MODEL_SEED,
    "feature_order": FEATURES,
    "target_transform": "log1p",
    "target_inverse": "expm1",
    "prediction_clipping": False,
    "scaler": "SVR only; fitted per bootstrap sample",
    "validation": "fixed, 4032 observations",
    "test_access": False,
    "software_versions": current_versions
}

PROTOCOL_HASH = sha256_object(protocol)

PROTOCOL_PATH = (
    REPORT_DIR / "cell02b_refitting_protocol.json"
)

if PROTOCOL_PATH.exists():

    require(
        load_json(PROTOCOL_PATH) == protocol,
        "Existing refitting protocol differs."
    )

else:

    with open(
        PROTOCOL_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            protocol,
            f,
            indent=2,
            ensure_ascii=False
        )


# =====================================================================
# 8. METRIC FUNCTION — ORIGINAL WH SCALE
# =====================================================================

def evaluate_predictions(prediction_wh):

    prediction_wh = np.asarray(
        prediction_wh,
        dtype=np.float64
    ).reshape(-1)

    require(
        prediction_wh.shape == (N_VAL,),
        "Validation prediction shape mismatch."
    )

    require(
        np.isfinite(prediction_wh).all(),
        "Nonfinite model predictions."
    )

    # No extra clipping or postprocessing.
    # TTR has already applied expm1.

    metrics = {
        "rmse_wh": float(
            np.sqrt(
                mean_squared_error(
                    y_validation_wh,
                    prediction_wh
                )
            )
        ),

        "mae_wh": float(
            mean_absolute_error(
                y_validation_wh,
                prediction_wh
            )
        ),

        "r2": float(
            r2_score(
                y_validation_wh,
                prediction_wh
            )
        )
    }

    require(
        all(np.isfinite(list(metrics.values()))),
        "Nonfinite validation metric."
    )

    return metrics


# =====================================================================
# 9. ATOMIC CHECKPOINT WRITING
# =====================================================================

def save_checkpoint(path, prediction_wh, metadata):

    require(
        not path.exists(),
        "Refusing to overwrite existing checkpoint."
    )

    temporary_path = path.with_name(
        path.name + ".temporary"
    )

    require(
        not temporary_path.exists(),
        f"Unresolved temporary checkpoint: "
        f"{temporary_path}"
    )

    # Save full validation predictions and metadata
    # in one compressed file, without fitted model.

    with open(temporary_path, "wb") as f:

        np.savez_compressed(
            f,
            prediction_wh=np.asarray(
                prediction_wh, dtype=np.float64
            ),
            metadata=np.array(
                canonical_json(metadata)
            )
        )

    os.replace(
        temporary_path,
        path
    )


def read_checkpoint(path, expected_identity):

    with np.load(
        path,
        allow_pickle=False
    ) as stored:

        require(
            set(stored.files)
            == {"prediction_wh", "metadata"},
            f"Checkpoint schema mismatch: {path}"
        )

        prediction = stored[
            "prediction_wh"
        ].copy()

        metadata = json.loads(
            stored["metadata"].item()
        )

    for name, expected in expected_identity.items():

        require(
            metadata.get(name) == expected,
            f"Checkpoint identity mismatch: "
            f"{path.name}, field={name}"
        )

    require(
        sha256_array(prediction)
        == metadata["prediction_sha256"],
        f"Prediction checksum mismatch: {path}"
    )

    recalculated = evaluate_predictions(
        prediction
    )

    for metric, value in recalculated.items():

        require(
            np.isclose(
                value,
                metadata[metric],
                rtol=1e-10,
                atol=1e-10
            ),
            f"Metric mismatch: {path}, {metric}"
        )

    return metadata


# =====================================================================
# 10. EXECUTE ALL 30 REPLICATIONS
# =====================================================================

print("=" * 76)
print("CELL 02B — FULL B30 BOOTSTRAP REFITTING")
print("=" * 76)

print("Training:", (N, len(FEATURES)))
print("Validation:", (N_VAL, len(FEATURES)))
print("Algorithms:", len(MODEL_ORDER))
print("Model settings: default + tuned")
print("Resampling conditions:", len(CONDITIONS))
print("Replications per condition:", B)
print("Total model fits:", 2520)
print("Model random seed:", MODEL_SEED)
print("Protocol SHA256:", PROTOCOL_HASH)

print("\nStarting full B30 execution...\n")

all_records = []
completed_new = 0
completed_existing = 0

for condition in CONDITIONS:

    item = index_registry[
        "checkpoints"
    ][condition]

    index_path = (
        INDEX_DIR / item["file"]
    )

    require(
        index_path.is_file(),
        f"Index checkpoint missing: {condition}"
    )

    require(
        sha256_file(index_path) == item["sha256"],
        f"Index checkpoint hash mismatch: {condition}"
    )

    with np.load(
        index_path,
        allow_pickle=False
    ) as index_file:

        index_matrix = index_file[
            "indices"
        ].copy()

    require(
        index_matrix.shape == (B, N),
        f"Incorrect index matrix: {condition}"
    )

    require(
        np.issubdtype(
            index_matrix.dtype,
            np.integer
        ),
        f"Noninteger indices: {condition}"
    )

    require(
        index_matrix.min() >= 0
        and index_matrix.max() < N,
        f"Invalid indices: {condition}"
    )

    condition_dir = (
        CHECKPOINT_DIR / condition
    )

    condition_dir.mkdir(
        parents=True, exist_ok=True
    )

    print(
        f"\nCONDITION: {condition}",
        flush=True
    )

    for replication in range(1, B + 1):

        indices = index_matrix[
            replication - 1
        ]

        index_hash = sha256_array(
            indices
        )

        # Each replication uses exactly the same
        # sampled rows for all 12 model configurations.

        X_boot = (
            X_train.iloc[indices]
            .reset_index(drop=True)
        )

        y_boot = (
            y_train_wh.iloc[indices]
            .reset_index(drop=True)
        )

        require(
            X_boot.shape == (N, 27)
            and len(y_boot) == N,
            "Bootstrap pair alignment mismatch."
        )

        replication_dir = (
            condition_dir
            / f"rep_{replication:03d}"
        )

        replication_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        rep_new = 0
        rep_existing = 0

        for algorithm in MODEL_ORDER:

            for setting in [
                "default",
                "tuned"
            ]:

                slug = MODEL_SLUGS[
                    algorithm
                ]

                output_path = (
                    replication_dir
                    / f"{slug}__{setting}.npz"
                )

                candidate_id = (
                    "default"
                    if setting == "default"
                    else candidate_ids[algorithm]
                )

                identity = {
                    "cell_version": CELL_VERSION,
                    "protocol_sha256": PROTOCOL_HASH,
                    "condition": condition,
                    "replication": replication,
                    "algorithm": algorithm,
                    "setting": setting,
                    "candidate_id": candidate_id,
                    "index_sha256": index_hash
                }

                # Resume safely from an existing
                # completed model checkpoint.

                if output_path.exists():

                    metadata = read_checkpoint(
                        output_path,
                        identity
                    )

                    rep_existing += 1
                    completed_existing += 1

                else:

                    model = make_frozen_model(
                        algorithm,
                        setting
                    )

                    started = time.perf_counter()

                    try:

                        model.fit(
                            X_boot,
                            y_boot
                        )

                        fit_seconds = (
                            time.perf_counter()
                            - started
                        )

                        prediction_wh = (
                            model.predict(
                                X_validation
                            )
                        )

                        prediction_wh = (
                            np.asarray(
                                prediction_wh,
                                dtype=np.float64
                            ).reshape(-1)
                        )

                        metrics = evaluate_predictions(
                            prediction_wh
                        )

                    except Exception as error:

                        print(
                            "\nFIT FAILED",
                            "condition=", condition,
                            "replication=", replication,
                            "algorithm=", algorithm,
                            "setting=", setting,
                            flush=True
                        )

                        raise RuntimeError(
                            "Bootstrap model refitting "
                            "stopped. Completed checkpoints "
                            "remain preserved."
                        ) from error

                    metadata = {
                        **identity,
                        **metrics,
                        "fit_seconds": float(
                            fit_seconds
                        ),
                        "sample_size": N,
                        "validation_size": N_VAL,
                        "model_seed": MODEL_SEED,
                        "target_scale": "Wh",
                        "preprocessing": "B_log",
                        "prediction_sha256": (
                            sha256_array(
                                prediction_wh
                            )
                        ),
                        "status": "PASS"
                    }

                    save_checkpoint(
                        output_path,
                        prediction_wh,
                        metadata
                    )

                    # Read back the saved output
                    # immediately before proceeding.

                    metadata = read_checkpoint(
                        output_path,
                        identity
                    )

                    rep_new += 1
                    completed_new += 1

                    # Release fitted estimator.
                    del model

                all_records.append({
                    "condition": condition,
                    "replication": replication,
                    "algorithm": algorithm,
                    "setting": setting,
                    "candidate_id": candidate_id,
                    "rmse_wh": metadata["rmse_wh"],
                    "mae_wh": metadata["mae_wh"],
                    "r2": metadata["r2"],
                    "fit_seconds": metadata[
                        "fit_seconds"
                    ],
                    "index_sha256": index_hash,
                    "prediction_sha256": metadata[
                        "prediction_sha256"
                    ],
                    "checkpoint_path": str(
                        output_path
                    ),
                    "status": "PASS"
                })

        require(
            rep_new + rep_existing == 12,
            "Incomplete replication."
        )

        print(
            f"{condition} | "
            f"rep {replication:02d}/{B} | "
            f"new={rep_new} | "
            f"resumed={rep_existing} | PASS",
            flush=True
        )


# =====================================================================
# 11. FULL EXPERIMENT AUDIT
# =====================================================================

results = pd.DataFrame(
    all_records
)

require(
    len(results) == 2520,
    "Incomplete B30 result registry."
)

require(
    results[[
        "condition",
        "replication",
        "algorithm",
        "setting"
    ]].duplicated().sum() == 0,
    "Duplicate experiment result keys."
)

require(
    results["status"].eq("PASS").all(),
    "Incomplete model fitting detected."
)

require(
    results[[
        "rmse_wh",
        "mae_wh",
        "r2",
        "fit_seconds"
    ]].apply(
        lambda col: np.isfinite(
            col.to_numpy(dtype=float)
        ).all()
    ).all(),
    "Nonfinite result values detected."
)

# Every group must contain 12 models sharing
# exactly the same outer resampling indices.

group_audit = (
    results.groupby(
        ["condition", "replication"]
    )
    .agg(
        model_fits=("algorithm", "size"),
        algorithms=("algorithm", "nunique"),
        settings=("setting", "nunique"),
        index_fingerprints=(
            "index_sha256",
            "nunique"
        )
    )
    .reset_index()
)

require(
    len(group_audit) == 7 * B,
    "Incorrect number of replication groups."
)

require(
    group_audit["model_fits"].eq(12).all(),
    "Missing model in a replication."
)

require(
    group_audit["algorithms"].eq(6).all()
    and group_audit["settings"].eq(2).all(),
    "Algorithm/setting mismatch."
)

require(
    group_audit[
        "index_fingerprints"
    ].eq(1).all(),
    "Models did not use identical indices "
    "within a replication."
)


# =====================================================================
# 12. SAVE COMPLETE B30 RESULTS
# =====================================================================

METRICS_PATH = (
    REPORT_DIR
    / "bootstrap_B30_all_replication_metrics.csv"
)

GROUP_AUDIT_PATH = (
    REPORT_DIR
    / "bootstrap_B30_replication_audit.csv"
)

MANIFEST_PATH = (
    REPORT_DIR
    / "bootstrap_B30_completion_manifest.json"
)


def atomic_csv_write(frame, path):

    temporary = path.with_name(
        path.name + ".temporary"
    )

    frame.to_csv(
        temporary,
        index=False,
        float_format="%.12g"
    )

    os.replace(
        temporary,
        path
    )


atomic_csv_write(
    results,
    METRICS_PATH
)

atomic_csv_write(
    group_audit,
    GROUP_AUDIT_PATH
)

completion_manifest = {
    "cell": "02B",
    "status": "PASS",
    "protocol_sha256": PROTOCOL_HASH,
    "conditions": CONDITIONS,
    "replications_per_condition": B,
    "algorithms": MODEL_ORDER,
    "model_settings": ["default", "tuned"],
    "total_model_fits": 2520,
    "new_fits_this_execution": completed_new,
    "resumed_existing_fits": completed_existing,
    "individual_prediction_files": len(results),
    "fixed_validation_size": N_VAL,
    "training_source": str(PATHS["train"]),
    "validation_source": str(PATHS["validation"]),
    "model_selection_source": str(
        PATHS["selection"]
    ),
    "test_accessed": False,
    "hyperparameter_tuning": False,
    "bootstrap_indices_regenerated": False,
    "outputs": {
        "metrics_csv": str(METRICS_PATH),
        "metrics_sha256": sha256_file(
            METRICS_PATH
        ),
        "replication_audit_csv": str(
            GROUP_AUDIT_PATH
        ),
        "replication_audit_sha256": (
            sha256_file(GROUP_AUDIT_PATH)
        )
    }
}

with open(
    MANIFEST_PATH.with_name(
        MANIFEST_PATH.name + ".temporary"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        completion_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

os.replace(
    MANIFEST_PATH.with_name(
        MANIFEST_PATH.name + ".temporary"
    ),
    MANIFEST_PATH
)


# =====================================================================
# 13. FINAL REPORT
# =====================================================================

print("\n" + "=" * 76)
print("CELL 02B — FULL B30 EXECUTION COMPLETED")
print("=" * 76)

print("\nA. EXECUTION SUMMARY")
print("Expected fits:", 2520)
print("Completed fits:", len(results))
print("New fits:", completed_new)
print("Resumed existing:", completed_existing)
print("Prediction checkpoints:", len(results))

print("\nB. CONDITION AUDIT")

condition_summary = (
    group_audit.groupby("condition")
    .agg(
        replications=("replication", "nunique"),
        model_fits=("model_fits", "sum"),
        shared_index_verified=(
            "index_fingerprints",
            lambda s: bool((s == 1).all())
        )
    )
    .reset_index()
)

display(condition_summary)

print("\nC. OUTPUT LOCATIONS")
print("Individual checkpoints:", CHECKPOINT_DIR)
print("Metrics:", METRICS_PATH)
print("Replication audit:", GROUP_AUDIT_PATH)
print("Completion manifest:", MANIFEST_PATH)

print("\nFINAL STATUS: PASS")
print("B30 completed: YES")
print("Bootstrap conditions: 7 / 7")
print("Model configurations: 12 / 12")
print("Validation predictions saved: YES")
print("Bootstrap indices regenerated: NO")
print("Hyperparameters retuned: NO")
print("Final test accessed: NO")
print("Figures generated: NO")


CELL 02B — FULL B30 BOOTSTRAP REFITTING
Training: (11850, 27)
Validation: (4032, 27)
Algorithms: 6
Model settings: default + tuned
Resampling conditions: 7
Replications per condition: 30
Total model fits: 2520
Model random seed: 42
Protocol SHA256: 0746b953192ec2cfc1a8f3025381f4aba8843456ae1db3ae8c2b9f0110d559f9

Starting full B30 execution...


CONDITION: iid
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004723 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5841
[LightGBM] [Info] Number of data points in the train set: 11850, number of used features: 27
[LightGBM] [Info] Start training from score 4.284295
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002772 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5841
[LightGBM] [Info] Number of data points in the train set: 11850, number of used features: 27
[Light

,condition,replications,model_fits,shared_index_verified
0,iid,30,360,True
1,mbb_L0072,30,360,True
2,mbb_L0144,30,360,True
3,mbb_L0432,30,360,True
4,stationary_L0072,30,360,True
5,stationary_L0144,30,360,True
6,stationary_L0432,30,360,True



C. OUTPUT LOCATIONS
Individual checkpoints: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B30/checkpoints
Metrics: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B30/reports/bootstrap_B30_all_replication_metrics.csv
Replication audit: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B30/reports/bootstrap_B30_replication_audit.csv
Completion manifest: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B30/reports/bootstrap_B30_completion_manifest.json

FINAL STATUS: PASS
B30 completed: YES
Bootstrap conditions: 7 / 7
Model configurations: 12 / 12
Validation predictions saved: YES
Bootstrap indices regenerated: NO
Hyperparameters retuned: NO
Final test accessed: NO
Figures generated: NO


## 02C — Ekstensi bootstrap B=30 → B=100

In [ ]:

# ================================================================
# NOTEBOOK 03 — CELL 02C
# CUMULATIVE B=100 BOOTSTRAP REFITTING
#
# Existing: B=30 (2,520 completed fits)
# New:      B=31..100 (5,880 additional fits)
# Total:    8,400 cumulative fits
#
# Frozen sources: Notebook 02, Cell 01, Cell 02B
# NO RETUNING | NO TEST ACCESS | NO REPEATED B30 FITTING
# ================================================================

import os
import json
import math
import time
import hashlib
import importlib.metadata

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor
)
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


# ================================================================
# 1. REGISTER FROZEN SOURCES AND OUTPUT DIRECTORY
# ================================================================

ROOT = Path(
    "/content/drive/MyDrive/Stability_Driven_Energy"
)

DATA_DIR = (
    ROOT / "02_preprocessed_data"
    / "temporal_partitions"
)

SPEC_DIR = (
    ROOT / "09_experiment_metadata"
    / "model_specification"
)

RUN_DIR = (
    ROOT / "03_model_development"
    / "grid_search" / "3540b29b8201"
)

INDEX30_DIR = (
    ROOT / "04_bootstrap" / "resampling_indices"
)

B30_DIR = (
    ROOT / "04_bootstrap" / "model_refitting_B30"
)

B100_DIR = (
    ROOT / "04_bootstrap" / "model_refitting_B100"
)

INDEX100_DIR = (
    B100_DIR / "resampling_indices_extension"
)

CHECK100_DIR = B100_DIR / "checkpoints"
REPORT100_DIR = B100_DIR / "reports"

for directory in [
    INDEX100_DIR,
    CHECK100_DIR,
    REPORT100_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

CELL_VERSION = "cell02c_cumulative_B100_v1"
PLAN_ID = "3540b29b8201"

B_OLD = 30
B_TARGET = 100
MODEL_SEED = 42

CONDITIONS = [
    "iid",
    "mbb_L0072",
    "mbb_L0144",
    "mbb_L0432",
    "stationary_L0072",
    "stationary_L0144",
    "stationary_L0432"
]

MODEL_CLASSES = {
    "Decision Tree": DecisionTreeRegressor,
    "Random Forest": RandomForestRegressor,
    "Bagging": BaggingRegressor,
    "SVR": SVR,
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor
}

MODEL_SLUGS = {
    "Decision Tree": "decision_tree",
    "Random Forest": "random_forest",
    "Bagging": "bagging",
    "SVR": "svr",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm"
}

MODELS = list(MODEL_CLASSES)
SETTINGS = ("default", "tuned")

PATHS = {
    "train": DATA_DIR / "B_train.csv",
    "validation": DATA_DIR / "B_validation.csv",

    "defaults": (
        SPEC_DIR / "default_estimator_parameters.json"
    ),

    "blueprints": (
        SPEC_DIR / "pipeline_blueprints.csv"
    ),

    "software": (
        SPEC_DIR / "software_environment.json"
    ),

    "selection": (
        RUN_DIR / "reports"
        / "final_tuned_model_selection.csv"
    ),

    "final_audit": (
        RUN_DIR / "reports"
        / "notebook02_final_audit.json"
    ),

    "plan": (
        INDEX30_DIR
        / "bootstrap_fixed_grid_plan.json"
    ),

    "index_registry": (
        INDEX30_DIR
        / "pilot_B30_checkpoint_registry.json"
    ),

    "b30_metrics": (
        B30_DIR / "reports"
        / "bootstrap_B30_all_replication_metrics.csv"
    ),

    "b30_replication_audit": (
        B30_DIR / "reports"
        / "bootstrap_B30_replication_audit.csv"
    ),

    "b30_manifest": (
        B30_DIR / "reports"
        / "bootstrap_B30_completion_manifest.json"
    ),

    "b30_protocol": (
        B30_DIR / "reports"
        / "cell02b_refitting_protocol.json"
    )
}


# ================================================================
# 2. UTILITY FUNCTIONS
# ================================================================

def ensure(condition, message):
    if not condition:
        raise RuntimeError(message)


def read_json(path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )


def sha_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha_array(values):
    return hashlib.sha256(
        np.ascontiguousarray(values).tobytes()
    ).hexdigest()


def canonical(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        allow_nan=False
    )


def sha_object(obj):
    return hashlib.sha256(
        canonical(obj).encode("utf-8")
    ).hexdigest()


def atomic_json(path, obj):
    temporary = path.with_name(
        path.name + ".temporary"
    )

    with open(
        temporary,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
            allow_nan=False
        )

    os.replace(temporary, path)


def atomic_csv(path, dataframe):
    temporary = path.with_name(
        path.name + ".temporary"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        float_format="%.12g"
    )

    os.replace(temporary, path)


# ================================================================
# 3. VERIFY NOTEBOOK 02 AND B30 ARTIFACTS
# ================================================================

for name, path in PATHS.items():
    ensure(
        path.is_file(),
        f"Missing frozen artifact: {name}: {path}"
    )

plan = read_json(PATHS["plan"])
registry30 = read_json(PATHS["index_registry"])
audit02 = read_json(PATHS["final_audit"])

b30_manifest = read_json(PATHS["b30_manifest"])
b30_protocol = read_json(PATHS["b30_protocol"])

defaults = read_json(PATHS["defaults"])
software = read_json(PATHS["software"])

selection = pd.read_csv(PATHS["selection"])
blueprints = pd.read_csv(PATHS["blueprints"])

ensure(
    audit02.get("status") == "PASS"
    and audit02.get("plan_id") == PLAN_ID
    and audit02.get("preprocessing") == "B_log",
    "Notebook 02 audit mismatch."
)

ensure(
    plan.get("experiment_version")
    == "bootstrap_fixed_grid_v1"
    and plan.get("notebook02_plan_id") == PLAN_ID
    and plan.get("preprocessing") == "B_log"
    and plan.get("feature_set") == "B"
    and plan.get("master_seed") == 42
    and plan.get("block_lengths") == [72, 144, 432]
    and plan.get("pilot_repetitions") == B_OLD,
    "Frozen Cell 01 protocol mismatch."
)

ensure(
    registry30.get("status") == "PASS"
    and registry30.get("pilot_B") == B_OLD
    and registry30.get("training_sha256")
    == plan["training_sha256"],
    "Cell 01 registry mismatch."
)

ensure(
    b30_manifest.get("status") == "PASS"
    and b30_manifest.get("total_model_fits") == 2520,
    "B30 completion manifest mismatch."
)

ensure(
    sha_object(b30_protocol)
    == b30_manifest["protocol_sha256"],
    "B30 protocol hash mismatch."
)

ensure(
    sha_file(PATHS["b30_metrics"])
    == b30_manifest["outputs"]["metrics_sha256"],
    "Original B30 metrics checksum mismatch."
)

ensure(
    sha_file(PATHS["b30_replication_audit"])
    == b30_manifest["outputs"][
        "replication_audit_sha256"
    ],
    "B30 replication audit checksum mismatch."
)

for artifact, hash_key in [
    ("train", "training_sha256"),
    ("validation", "validation_sha256"),
    ("selection", "model_selection_sha256")
]:
    ensure(
        sha_file(PATHS[artifact]) == plan[hash_key],
        f"Frozen {artifact} changed."
    )


# ================================================================
# 4. VERIFY SOFTWARE VERSIONS
# ================================================================

PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "scikit_learn": "scikit-learn",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm"
}

for label, package in PACKAGES.items():
    current = importlib.metadata.version(package)

    ensure(
        str(current) == str(software[label]),
        f"{label} version differs from Notebook 02."
    )


# ================================================================
# 5. LOAD FROZEN TRAINING AND VALIDATION
# ================================================================

train = pd.read_csv(
    PATHS["train"],
    parse_dates=["date"]
)

validation = pd.read_csv(
    PATHS["validation"],
    parse_dates=["date"]
)

FEATURES = plan["predictor_order"]

ensure(
    len(train) == 11850
    and len(validation) == 4032
    and len(FEATURES) == 27,
    "Frozen dataset dimensions mismatch."
)

ensure(
    [
        c for c in train.columns
        if c not in ("date", "Appliances")
    ] == FEATURES,
    "Training predictor order mismatch."
)

ensure(
    [
        c for c in validation.columns
        if c not in ("date", "Appliances")
    ] == FEATURES,
    "Validation predictor order mismatch."
)

ensure(
    train["date"].is_monotonic_increasing
    and validation["date"].is_monotonic_increasing
    and train["date"].iloc[-1]
    < validation["date"].iloc[0],
    "Temporal ordering mismatch."
)

X_train = train[FEATURES].copy()
y_train = train["Appliances"].copy()

X_validation = validation[FEATURES].copy()

y_validation = validation[
    "Appliances"
].to_numpy(dtype=np.float64)

ensure(
    np.isfinite(
        X_train.to_numpy(dtype=float)
    ).all()
    and np.isfinite(
        X_validation.to_numpy(dtype=float)
    ).all()
    and np.isfinite(
        y_train.to_numpy(dtype=float)
    ).all()
    and np.isfinite(y_validation).all(),
    "Nonfinite frozen input values."
)

ensure(
    y_train.ge(0).all()
    and np.all(y_validation >= 0),
    "Negative target values."
)

N = len(train)
N_VAL = len(validation)


# ================================================================
# 6. REBUILD EXACT FROZEN MODEL SPECIFICATIONS
# ================================================================

ensure(
    set(defaults) == set(MODELS)
    and len(selection) == 6
    and selection["algorithm"].is_unique
    and set(selection["algorithm"]) == set(MODELS),
    "Model registration mismatch."
)

candidate_ids = dict(
    zip(
        selection["algorithm"].astype(str),
        selection["candidate_id"].astype(str)
    )
)

tuned_parameters = {
    str(row.algorithm): json.loads(
        row.parameters_json
    )
    for row in selection.itertuples(index=False)
}

ensure(
    plan["model_selection"] == candidate_ids,
    "Frozen tuned candidate IDs changed."
)

b_log = blueprints.loc[
    blueprints["configuration"] == "B_log"
]

ensure(
    len(b_log) == 6
    and set(b_log["algorithm"]) == set(MODELS)
    and b_log["n_features"].eq(27).all()
    and b_log["target_treatment"].eq("log1p").all(),
    "B_log blueprint mismatch."
)

for row in b_log.itertuples(index=False):
    scaler = (
        "none"
        if pd.isna(row.scaler)
        else str(row.scaler).lower()
    )

    expected_scaler = (
        "standardscaler"
        if row.algorithm == "SVR"
        else "none"
    )

    ensure(
        scaler == expected_scaler
        and str(row.example_parameter_key).startswith(
            "regressor__model__"
        ),
        f"Pipeline blueprint mismatch: {row.algorithm}"
    )


def build_model(algorithm, setting):

    # Same model factory as Cell 02B.
    estimator = MODEL_CLASSES[algorithm](
        **dict(defaults[algorithm])
    )

    if setting == "tuned":
        if algorithm == "Bagging":
            estimator.set_params(
                estimator=DecisionTreeRegressor(
                    random_state=MODEL_SEED
                )
            )

        estimator.set_params(
            **dict(tuned_parameters[algorithm])
        )

    steps = []

    if algorithm == "SVR":
        steps.append(
            ("scaler", StandardScaler())
        )

    steps.append(("model", estimator))

    return TransformedTargetRegressor(
        regressor=Pipeline(steps),
        func=np.log1p,
        inverse_func=np.expm1
    )


def evaluate_prediction(prediction):

    prediction = np.asarray(
        prediction,
        dtype=np.float64
    ).reshape(-1)

    ensure(
        prediction.shape == (N_VAL,)
        and np.isfinite(prediction).all(),
        "Invalid validation predictions."
    )

    return {
        "rmse_wh": float(
            np.sqrt(
                mean_squared_error(
                    y_validation,
                    prediction
                )
            )
        ),
        "mae_wh": float(
            mean_absolute_error(
                y_validation,
                prediction
            )
        ),
        "r2": float(
            r2_score(
                y_validation,
                prediction
            )
        )
    }


# ================================================================
# 7. VERIFY ALL ORIGINAL B30 RESULTS
# ================================================================

b30 = pd.read_csv(PATHS["b30_metrics"])

audit30 = pd.read_csv(
    PATHS["b30_replication_audit"]
)

KEYS = [
    "condition",
    "replication",
    "algorithm",
    "setting"
]

ensure(
    len(b30) == 2520
    and not b30.duplicated(KEYS).any()
    and b30["status"].eq("PASS").all(),
    "B30 metrics structure invalid."
)

ensure(
    len(audit30) == 210
    and audit30["model_fits"].eq(12).all()
    and audit30["algorithms"].eq(6).all()
    and audit30["settings"].eq(2).all()
    and audit30["index_fingerprints"].eq(1).all(),
    "B30 replication audit invalid."
)

ensure(
    set(b30["condition"]) == set(CONDITIONS)
    and b30["replication"].between(1, 30).all()
    and set(b30["algorithm"]) == set(MODELS)
    and set(b30["setting"]) == set(SETTINGS)
    and b30.groupby(
        ["condition", "replication"]
    ).size().eq(12).all()
    and b30.groupby(
        ["condition", "replication"]
    )["index_sha256"].nunique().eq(1).all(),
    "B30 condition/replication mismatch."
)

for row in b30.itertuples(index=False):

    expected_id = (
        "default"
        if row.setting == "default"
        else candidate_ids[row.algorithm]
    )

    ensure(
        row.candidate_id == expected_id,
        "B30 selected model ID mismatch."
    )


# ================================================================
# 8. AUDIT ORIGINAL B30 CHECKPOINT INTEGRITY
# ================================================================

B30_RECEIPT_PATH = (
    REPORT100_DIR
    / "B30_individual_checkpoint_integrity_receipt.json"
)

receipt_identity = {
    "status": "PASS",
    "metrics_sha256": sha_file(
        PATHS["b30_metrics"]
    ),
    "b30_manifest_sha256": sha_file(
        PATHS["b30_manifest"]
    ),
    "b30_protocol_sha256": sha_object(
        b30_protocol
    ),
    "index_registry_sha256": sha_file(
        PATHS["index_registry"]
    ),
    "checkpoint_files_verified": 2520
}

# Verify all original bootstrap index hashes.

for condition in CONDITIONS:

    item = registry30[
        "checkpoints"
    ][condition]

    index_path = INDEX30_DIR / item["file"]

    ensure(
        index_path.is_file()
        and sha_file(index_path) == item["sha256"],
        f"B30 index checkpoint changed: {condition}"
    )

    with np.load(
        index_path,
        allow_pickle=False
    ) as source:
        original_indices = source["indices"]

    ensure(
        original_indices.shape == (30, N),
        "B30 index matrix shape mismatch."
    )

    for rep in range(1, 31):

        actual_hash = sha_array(
            original_indices[rep - 1]
        )

        recorded = b30.loc[
            (b30["condition"] == condition)
            & (b30["replication"] == rep),
            "index_sha256"
        ].unique()

        ensure(
            len(recorded) == 1
            and recorded[0] == actual_hash,
            f"B30 index fingerprint mismatch: "
            f"{condition}, rep {rep}"
        )


if B30_RECEIPT_PATH.is_file():

    ensure(
        read_json(B30_RECEIPT_PATH)
        == receipt_identity,
        "Previous B30 integrity receipt differs."
    )

    print(
        "B30 individual checkpoint audit: "
        "VERIFIED CACHED RECEIPT"
    )

else:

    print(
        "Verifying all 2,520 B30 prediction "
        "checkpoints (one-time audit)...",
        flush=True
    )

    for row in b30.itertuples(index=False):

        expected_path = (
            B30_DIR / "checkpoints"
            / row.condition
            / f"rep_{row.replication:03d}"
            / (
                f"{MODEL_SLUGS[row.algorithm]}"
                f"__{row.setting}.npz"
            )
        )

        ensure(
            str(expected_path) == row.checkpoint_path
            and expected_path.is_file(),
            f"Missing/unexpected B30 checkpoint: "
            f"{expected_path}"
        )

        with np.load(
            expected_path,
            allow_pickle=False
        ) as stored:

            ensure(
                set(stored.files)
                == {"prediction_wh", "metadata"},
                "B30 checkpoint schema mismatch."
            )

            prediction = stored[
                "prediction_wh"
            ]

            metadata = json.loads(
                stored["metadata"].item()
            )

        expected_metadata = {
            "cell_version": "cell02b_full_B30_v1",
            "protocol_sha256": receipt_identity[
                "b30_protocol_sha256"
            ],
            "condition": row.condition,
            "replication": int(row.replication),
            "algorithm": row.algorithm,
            "setting": row.setting,
            "candidate_id": row.candidate_id,
            "index_sha256": row.index_sha256
        }

        for key, value in expected_metadata.items():

            ensure(
                metadata.get(key) == value,
                f"B30 metadata mismatch: "
                f"{expected_path} [{key}]"
            )

        ensure(
            sha_array(prediction)
            == metadata["prediction_sha256"]
            == row.prediction_sha256,
            f"B30 prediction checksum mismatch: "
            f"{expected_path}"
        )

        metrics = evaluate_prediction(prediction)

        for key in ["rmse_wh", "mae_wh", "r2"]:

            ensure(
                np.isclose(
                    metrics[key],
                    float(getattr(row, key)),
                    atol=1e-7,
                    rtol=1e-10
                )
                and np.isclose(
                    metrics[key],
                    float(metadata[key]),
                    atol=1e-7,
                    rtol=1e-10
                ),
                f"B30 metric mismatch: "
                f"{expected_path} [{key}]"
            )

    atomic_json(
        B30_RECEIPT_PATH,
        receipt_identity
    )

    print(
        "B30 individual checkpoint audit: "
        "2,520 / 2,520 PASS",
        flush=True
    )


# ================================================================
# 9. REPRODUCE CELL 01 SEED AND INDEX GENERATORS
# ================================================================

def seed_for(method, length, replication):

    key = (
        f"{plan['experiment_version']}|"
        f"{PLAN_ID}|"
        f"{plan['training_sha256']}|"
        f"{plan['master_seed']}|"
        f"{method}|{length}|{replication}"
    )

    digest = hashlib.sha256(
        key.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:16],
        "little"
    )


def generate_indices(method, length, replication):

    rng = np.random.default_rng(
        seed_for(method, length, replication)
    )

    if method == "iid":

        return rng.integers(
            low=0,
            high=N,
            size=N,
            dtype=np.int32
        )

    if method == "mbb":

        ell = int(length)

        starts = rng.integers(
            low=0,
            high=N - ell + 1,
            size=math.ceil(N / ell)
        )

        output = np.empty(
            N,
            dtype=np.int32
        )

        cursor = 0

        for start in starts:

            used = min(
                ell,
                N - cursor
            )

            output[
                cursor:cursor + used
            ] = np.arange(
                int(start),
                int(start) + used
            )

            cursor += used

        ensure(
            cursor == N,
            "MBB generation incomplete."
        )

        return output

    if method == "stationary":

        p = 1.0 / float(length)

        output = np.empty(
            N,
            dtype=np.int32
        )

        cursor = 0

        while cursor < N:

            raw_length = int(
                rng.geometric(p)
            )

            start = int(
                rng.integers(0, N)
            )

            used = min(
                raw_length,
                N - cursor
            )

            output[
                cursor:cursor + used
            ] = (
                start + np.arange(used)
            ) % N

            cursor += used

        return output

    raise ValueError(
        f"Unknown bootstrap method: {method}"
    )


def decode_condition(condition):

    if condition == "iid":
        return "iid", None

    method, digits = condition.rsplit(
        "_L", 1
    )

    return method, int(digits)


# ================================================================
# 10. GENERATE ONLY REPLICATIONS 31–100
# ================================================================

extension_indices = {}
index100_registry = {}

for condition in CONDITIONS:

    method, length = decode_condition(
        condition
    )

    # Replay original indices to verify
    # that the generator/seed rule is unchanged.

    old_path = (
        INDEX30_DIR
        / registry30["checkpoints"][condition]["file"]
    )

    with np.load(
        old_path,
        allow_pickle=False
    ) as source:

        old_indices = source["indices"]

    for rep in (1, 30):

        replay = generate_indices(
            method,
            length,
            rep
        )

        ensure(
            np.array_equal(
                replay,
                old_indices[rep - 1]
            ),
            f"Cell 01 seed replay mismatch: "
            f"{condition}, rep {rep}"
        )

    matrix = np.stack(
        [
            generate_indices(
                method,
                length,
                rep
            )
            for rep in range(31, 101)
        ],
        axis=0
    )

    ensure(
        matrix.shape == (70, N)
        and np.issubdtype(
            matrix.dtype,
            np.integer
        )
        and matrix.min() >= 0
        and matrix.max() < N,
        f"Invalid extension indices: {condition}"
    )

    index_path = (
        INDEX100_DIR
        / f"{condition}_B0031_0100.npz"
    )

    if index_path.is_file():

        with np.load(
            index_path,
            allow_pickle=False
        ) as existing:

            ensure(
                set(existing.files) == {"indices"}
                and np.array_equal(
                    existing["indices"],
                    matrix
                ),
                f"Existing B100 indices differ: "
                f"{condition}"
            )

    else:

        temporary = index_path.with_name(
            index_path.name + ".temporary"
        )

        with open(temporary, "wb") as f:

            np.savez_compressed(
                f,
                indices=matrix
            )

        os.replace(
            temporary,
            index_path
        )

    extension_indices[condition] = matrix

    index100_registry[condition] = {
        "file": index_path.name,
        "sha256": sha_file(index_path)
    }


EXT_REGISTRY_PATH = (
    INDEX100_DIR
    / "B0031_0100_checkpoint_registry.json"
)

extension_registry = {
    "experiment_version": plan[
        "experiment_version"
    ],
    "source_training_sha256": plan[
        "training_sha256"
    ],
    "replications": list(range(31, 101)),
    "checkpoints": index100_registry,
    "status": "PASS"
}

if EXT_REGISTRY_PATH.is_file():

    ensure(
        read_json(EXT_REGISTRY_PATH)
        == extension_registry,
        "B100 index registry differs."
    )

else:

    atomic_json(
        EXT_REGISTRY_PATH,
        extension_registry
    )


# ================================================================
# 11. FREEZE B100 CONTINUATION PROTOCOL
# ================================================================

protocol = {
    "cell_version": CELL_VERSION,

    "base_cell02b_protocol_sha256": (
        sha_object(b30_protocol)
    ),

    "plan_sha256": sha_file(PATHS["plan"]),

    "source_training_sha256": plan[
        "training_sha256"
    ],

    "source_validation_sha256": plan[
        "validation_sha256"
    ],

    "model_selection_sha256": plan[
        "model_selection_sha256"
    ],

    "default_parameters_sha256": sha_file(
        PATHS["defaults"]
    ),

    "pipeline_blueprints_sha256": sha_file(
        PATHS["blueprints"]
    ),

    "B30_metrics_sha256": sha_file(
        PATHS["b30_metrics"]
    ),

    "extension_indices_registry_sha256": (
        sha_file(EXT_REGISTRY_PATH)
    ),

    "algorithm_order": MODELS,
    "conditions": CONDITIONS,
    "settings": list(SETTINGS),
    "model_seed": MODEL_SEED,

    "additional_reps": [31, 100],
    "cumulative_B": 100,

    "validation_rows": N_VAL,
    "target_scale": "Wh",

    "target_fit_transform": "log1p",
    "inverse_prediction_transform": "expm1",

    "no_clipping": True,
    "no_test_access": True
}

PROTOCOL_HASH = sha_object(
    protocol
)

PROTOCOL_PATH = (
    REPORT100_DIR
    / "cell02c_B100_extension_protocol.json"
)

if PROTOCOL_PATH.is_file():

    ensure(
        read_json(PROTOCOL_PATH) == protocol,
        "Existing B100 protocol differs."
    )

else:

    atomic_json(
        PROTOCOL_PATH,
        protocol
    )


# ================================================================
# 12. CHECKPOINT SAVE/RESUME FUNCTIONS
# ================================================================

def checkpoint_identity(
    condition,
    replication,
    algorithm,
    setting,
    index_hash
):

    return {
        "cell_version": CELL_VERSION,

        "protocol_sha256": PROTOCOL_HASH,

        "condition": condition,
        "replication": replication,

        "algorithm": algorithm,
        "setting": setting,

        "candidate_id": (
            "default"
            if setting == "default"
            else candidate_ids[algorithm]
        ),

        "index_sha256": index_hash
    }


def read_checkpoint(path, identity):

    with np.load(
        path,
        allow_pickle=False
    ) as stored:

        ensure(
            set(stored.files)
            == {"prediction_wh", "metadata"},
            f"Checkpoint schema mismatch: {path}"
        )

        prediction = stored[
            "prediction_wh"
        ].copy()

        metadata = json.loads(
            stored["metadata"].item()
        )

    for key, value in identity.items():

        ensure(
            metadata.get(key) == value,
            f"Checkpoint identity mismatch: "
            f"{path} [{key}]"
        )

    ensure(
        sha_array(prediction)
        == metadata.get("prediction_sha256"),
        f"Prediction SHA mismatch: {path}"
    )

    metrics = evaluate_prediction(
        prediction
    )

    for key, value in metrics.items():

        ensure(
            np.isclose(
                value,
                metadata[key],
                rtol=1e-10,
                atol=1e-8
            ),
            f"Checkpoint metric mismatch: "
            f"{path} [{key}]"
        )

    return metadata


def save_checkpoint(
    path,
    prediction,
    metadata
):

    ensure(
        not path.exists(),
        f"Refusing checkpoint overwrite: {path}"
    )

    temporary = path.with_name(
        path.name + ".temporary"
    )

    # A previous interrupted run may leave
    # an incomplete temporary file.

    if temporary.exists():
        temporary.unlink()

    with open(temporary, "wb") as f:

        np.savez_compressed(
            f,
            prediction_wh=np.asarray(
                prediction,
                dtype=np.float64
            ),
            metadata=np.array(
                canonical(metadata)
            )
        )

    os.replace(
        temporary,
        path
    )


# ================================================================
# 13. EXECUTE 5,880 ADDITIONAL FITS
# ================================================================

print("=" * 76)
print("CELL 02C — B=100 CUMULATIVE REFITTING")
print("=" * 76)

print("Original B30: 2,520 models preserved")
print("Additional: 5,880 fits, repetitions 31–100")
print("Target cumulative: 8,400 fits")

print("Original indices: VERIFIED")
print("New indices: 7 conditions x 70 repetitions")

print(
    "Extension protocol SHA256:",
    PROTOCOL_HASH
)

records_new = []

new_fit_count = 0
resumed_fit_count = 0

for condition in CONDITIONS:

    index_matrix = extension_indices[
        condition
    ]

    print(
        f"\nCONDITION: {condition}",
        flush=True
    )

    for replication in range(31, 101):

        indices = index_matrix[
            replication - 31
        ]

        index_hash = sha_array(
            indices
        )

        # Identical sampled rows for all 12 models.

        X_boot = (
            X_train.iloc[indices]
            .reset_index(drop=True)
        )

        y_boot = (
            y_train.iloc[indices]
            .reset_index(drop=True)
        )

        ensure(
            X_boot.shape == (N, 27)
            and len(y_boot) == N,
            "Bootstrap feature-target pair mismatch."
        )

        replication_dir = (
            CHECK100_DIR
            / condition
            / f"rep_{replication:03d}"
        )

        replication_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        rep_new = 0
        rep_resumed = 0

        for algorithm in MODELS:

            for setting in SETTINGS:

                output_path = (
                    replication_dir
                    / (
                        f"{MODEL_SLUGS[algorithm]}"
                        f"__{setting}.npz"
                    )
                )

                identity = checkpoint_identity(
                    condition,
                    replication,
                    algorithm,
                    setting,
                    index_hash
                )

                if output_path.is_file():

                    metadata = read_checkpoint(
                        output_path,
                        identity
                    )

                    rep_resumed += 1
                    resumed_fit_count += 1

                else:

                    model = build_model(
                        algorithm,
                        setting
                    )

                    started = time.perf_counter()

                    try:

                        model.fit(
                            X_boot,
                            y_boot
                        )

                        fit_seconds = (
                            time.perf_counter()
                            - started
                        )

                        prediction = np.asarray(
                            model.predict(
                                X_validation
                            ),
                            dtype=np.float64
                        ).reshape(-1)

                        metrics = evaluate_prediction(
                            prediction
                        )

                    except Exception as error:

                        raise RuntimeError(
                            f"FIT FAILED: {condition}, "
                            f"rep {replication}, "
                            f"{algorithm}, {setting}. "
                            "Earlier completed checkpoints "
                            "are preserved."
                        ) from error

                    metadata = {
                        **identity,
                        **metrics,

                        "fit_seconds": float(
                            fit_seconds
                        ),

                        "sample_size": N,
                        "validation_size": N_VAL,

                        "model_seed": MODEL_SEED,
                        "target_scale": "Wh",
                        "preprocessing": "B_log",

                        "prediction_sha256": (
                            sha_array(prediction)
                        ),

                        "status": "PASS"
                    }

                    save_checkpoint(
                        output_path,
                        prediction,
                        metadata
                    )

                    metadata = read_checkpoint(
                        output_path,
                        identity
                    )

                    del model

                    rep_new += 1
                    new_fit_count += 1

                records_new.append({
                    "condition": condition,
                    "replication": replication,

                    "algorithm": algorithm,
                    "setting": setting,

                    "candidate_id": metadata[
                        "candidate_id"
                    ],

                    "rmse_wh": metadata["rmse_wh"],
                    "mae_wh": metadata["mae_wh"],
                    "r2": metadata["r2"],

                    "fit_seconds": metadata[
                        "fit_seconds"
                    ],

                    "index_sha256": index_hash,

                    "prediction_sha256": metadata[
                        "prediction_sha256"
                    ],

                    "checkpoint_path": str(
                        output_path
                    ),

                    "status": "PASS"
                })

        ensure(
            rep_new + rep_resumed == 12,
            "Incomplete replication."
        )

        print(
            f"{condition} | "
            f"rep {replication:03d}/100 | "
            f"new={rep_new} | "
            f"resumed={rep_resumed} | PASS",
            flush=True
        )


# ================================================================
# 14. COMBINE ORIGINAL B30 + NEW B31–100
# ================================================================

new_df = pd.DataFrame(
    records_new,
    columns=b30.columns
)

ensure(
    len(new_df) == 5880
    and not new_df.duplicated(KEYS).any()
    and new_df["replication"].between(
        31, 100
    ).all(),
    "B100 extension incomplete."
)

combined = pd.concat(
    [b30, new_df],
    ignore_index=True
)

ensure(
    len(combined) == 8400
    and not combined.duplicated(KEYS).any()
    and combined["status"].eq("PASS").all()
    and set(combined["condition"]) == set(CONDITIONS)
    and combined.groupby(
        "condition"
    ).size().eq(1200).all(),
    "Cumulative B100 registry invalid."
)


# ================================================================
# 15. VERIFY PAIRED MODEL RESAMPLING
# ================================================================

summary = (
    combined.groupby(
        ["condition", "replication"],
        sort=False
    )
    .agg(
        model_fits=("algorithm", "size"),
        algorithms=("algorithm", "nunique"),
        settings=("setting", "nunique"),

        index_fingerprints=(
            "index_sha256",
            "nunique"
        )
    )
    .reset_index()
)

ensure(
    len(summary) == 700
    and summary["model_fits"].eq(12).all()
    and summary["algorithms"].eq(6).all()
    and summary["settings"].eq(2).all()
    and summary["index_fingerprints"].eq(1).all(),
    "Cumulative resampling pairing audit failed."
)


# ================================================================
# 16. STABLE OUTPUT ORDER
# ================================================================

combined["_condition_order"] = (
    combined["condition"].map(
        {
            condition: i
            for i, condition in enumerate(CONDITIONS)
        }
    )
)

combined["_algorithm_order"] = (
    combined["algorithm"].map(
        {
            algorithm: i
            for i, algorithm in enumerate(MODELS)
        }
    )
)

combined["_setting_order"] = (
    combined["setting"].map({
        "default": 0,
        "tuned": 1
    })
)

combined = (
    combined.sort_values(
        [
            "_condition_order",
            "replication",
            "_algorithm_order",
            "_setting_order"
        ],
        kind="stable"
    )
    .drop(
        columns=[
            "_condition_order",
            "_algorithm_order",
            "_setting_order"
        ]
    )
    .reset_index(drop=True)
)


# ================================================================
# 17. SAVE CUMULATIVE B100 RESULTS
# ================================================================

METRICS100_PATH = (
    REPORT100_DIR
    / "bootstrap_B100_all_replication_metrics.csv"
)

AUDIT100_PATH = (
    REPORT100_DIR
    / "bootstrap_B100_replication_audit.csv"
)

MANIFEST100_PATH = (
    REPORT100_DIR
    / "bootstrap_B100_completion_manifest.json"
)

atomic_csv(
    METRICS100_PATH,
    combined
)

atomic_csv(
    AUDIT100_PATH,
    summary
)

completion_manifest = {
    "cell": "02C",
    "status": "PASS",
    "cumulative_B": 100,

    "original_B30_fits_reused": 2520,

    "new_replications_31_to_100_fits": 5880,

    "new_fits_in_this_execution": (
        new_fit_count
    ),

    "already_present_extension_fits": (
        resumed_fit_count
    ),

    "total_fits": 8400,

    "prediction_checkpoints_old": 2520,
    "prediction_checkpoints_new": 5880,

    "conditions": CONDITIONS,
    "validation_rows": N_VAL,

    "protocol_sha256": PROTOCOL_HASH,

    "source_b30_metrics_sha256": (
        receipt_identity["metrics_sha256"]
    ),

    "test_accessed": False,
    "hyperparameters_retuned": False,

    "repetitions_1_to_30_refitted": False,

    "outputs": {
        "metrics_csv": str(
            METRICS100_PATH
        ),

        "metrics_sha256": sha_file(
            METRICS100_PATH
        ),

        "audit_csv": str(
            AUDIT100_PATH
        ),

        "audit_sha256": sha_file(
            AUDIT100_PATH
        )
    }
}

atomic_json(
    MANIFEST100_PATH,
    completion_manifest
)


# ================================================================
# 18. FINAL REPORT
# ================================================================

condition_summary = (
    summary.groupby(
        "condition",
        sort=False
    )
    .agg(
        repetitions=(
            "replication",
            "nunique"
        ),

        model_fits=(
            "model_fits",
            "sum"
        ),

        same_index_for_12_models=(
            "index_fingerprints",
            lambda x: bool(
                (x == 1).all()
            )
        )
    )
    .reset_index()
)

print("\n" + "=" * 76)
print("CELL 02C — CUMULATIVE B100 COMPLETED")
print("=" * 76)

print("\nA. EXECUTION SUMMARY")

print("Original B30 fits reused:", 2520)
print("New B31–100 fits:", new_fit_count)
print("Resumed B31–100 fits:", resumed_fit_count)
print("Cumulative fitting records:", len(combined))

print("\nB. CONDITION AUDIT")

display(condition_summary)

print("\nC. OUTPUT FILES")

print("Metrics CSV:", METRICS100_PATH)
print("Replication audit:", AUDIT100_PATH)
print("Completion manifest:", MANIFEST100_PATH)

print("\nFINAL STATUS: PASS")
print("Original B30 refitted: NO")
print("Hyperparameters retuned: NO")
print("Final test accessed: NO")
print("Figures generated: NO")


B30 individual checkpoint audit: VERIFIED CACHED RECEIPT
CELL 02C — B=100 CUMULATIVE REFITTING
Original B30: 2,520 models preserved
Additional: 5,880 fits, repetitions 31–100
Target cumulative: 8,400 fits
Original indices: VERIFIED
New indices: 7 conditions x 70 repetitions
Extension protocol SHA256: 27c719a66ecf2a2669c4a82046d4c1ecb3b5a91d0b8bb53f3e07e443051ca978

CONDITION: iid
iid | rep 031/100 | new=0 | resumed=12 | PASS
iid | rep 032/100 | new=0 | resumed=12 | PASS
iid | rep 033/100 | new=0 | resumed=12 | PASS
iid | rep 034/100 | new=0 | resumed=12 | PASS
iid | rep 035/100 | new=0 | resumed=12 | PASS
iid | rep 036/100 | new=0 | resumed=12 | PASS
iid | rep 037/100 | new=0 | resumed=12 | PASS
iid | rep 038/100 | new=0 | resumed=12 | PASS
iid | rep 039/100 | new=0 | resumed=12 | PASS
iid | rep 040/100 | new=0 | resumed=12 | PASS
iid | rep 041/100 | new=0 | resumed=12 | PASS
iid | rep 042/100 | new=0 | resumed=12 | PASS
iid | rep 043/100 | new=0 | resumed=12 | PASS
iid | rep 044/100 |

,condition,repetitions,model_fits,same_index_for_12_models
0,iid,100,1200,True
1,mbb_L0072,100,1200,True
2,mbb_L0144,100,1200,True
3,mbb_L0432,100,1200,True
4,stationary_L0072,100,1200,True
5,stationary_L0144,100,1200,True
6,stationary_L0432,100,1200,True



C. OUTPUT FILES
Metrics CSV: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B100/reports/bootstrap_B100_all_replication_metrics.csv
Replication audit: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B100/reports/bootstrap_B100_replication_audit.csv
Completion manifest: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B100/reports/bootstrap_B100_completion_manifest.json

FINAL STATUS: PASS
Original B30 refitted: NO
Hyperparameters retuned: NO
Final test accessed: NO
Figures generated: NO


## 02D — B=100 → B=300 kumulatif

In [ ]:

# ================================================================
# NOTEBOOK 03 — CELL 02D
# CUMULATIVE B=300 BOOTSTRAP MODEL REFITTING
#
# Previous B=100:  8,400 fits, reused
# New reps 101–300: 16,800 fits
# Cumulative total: 25,200 fits
#
# NO RETUNING | NO OLD REFITTING | NO TEST ACCESS
# ================================================================

import os
import json
import math
import time
import hashlib
import importlib.metadata
import contextlib
import io

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor
)
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from IPython.display import display


# ================================================================
# 1. FROZEN PATHS
# ================================================================

ROOT = Path(
    "/content/drive/MyDrive/Stability_Driven_Energy"
)

DATA = ROOT / "02_preprocessed_data" / "temporal_partitions"

SPEC = ROOT / "09_experiment_metadata" / "model_specification"

TUNE = (
    ROOT / "03_model_development"
    / "grid_search" / "3540b29b8201" / "reports"
)

IDX30 = ROOT / "04_bootstrap" / "resampling_indices"
B30 = ROOT / "04_bootstrap" / "model_refitting_B30"
B100 = ROOT / "04_bootstrap" / "model_refitting_B100"
B300 = ROOT / "04_bootstrap" / "model_refitting_B300"

IDX300 = B300 / "resampling_indices_extension"
CHECK300 = B300 / "checkpoints"
REPORT300 = B300 / "reports"

for folder in (IDX300, CHECK300, REPORT300):
    folder.mkdir(parents=True, exist_ok=True)

FILES = {
    "train": DATA / "B_train.csv",
    "validation": DATA / "B_validation.csv",
    "defaults": SPEC / "default_estimator_parameters.json",
    "blueprints": SPEC / "pipeline_blueprints.csv",
    "software": SPEC / "software_environment.json",
    "selection": TUNE / "final_tuned_model_selection.csv",
    "audit02": TUNE / "notebook02_final_audit.json",
    "plan": IDX30 / "bootstrap_fixed_grid_plan.json",
    "index30": IDX30 / "pilot_B30_checkpoint_registry.json",
    "index100": (
        B100 / "resampling_indices_extension"
        / "B0031_0100_checkpoint_registry.json"
    ),
    "protocol100": (
        B100 / "reports"
        / "cell02c_B100_extension_protocol.json"
    ),
    "manifest100": (
        B100 / "reports"
        / "bootstrap_B100_completion_manifest.json"
    ),
    "metrics100": (
        B100 / "reports"
        / "bootstrap_B100_all_replication_metrics.csv"
    ),
    "audit100": (
        B100 / "reports"
        / "bootstrap_B100_replication_audit.csv"
    ),
    "metrics30": (
        B30 / "reports"
        / "bootstrap_B30_all_replication_metrics.csv"
    )
}


# ================================================================
# 2. CHECKSUM AND ATOMIC FILE HELPERS
# ================================================================

def need(condition, message):
    if not condition:
        raise RuntimeError(message)


def readj(path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )


def sha_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha_array(values):
    return hashlib.sha256(
        np.ascontiguousarray(values).tobytes()
    ).hexdigest()


def canonical(value):
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False
    )


def sha_obj(value):
    return hashlib.sha256(
        canonical(value).encode("utf-8")
    ).hexdigest()


def write_json(path, value):
    temporary = path.with_name(
        path.name + ".temporary"
    )

    with temporary.open("w", encoding="utf-8") as file:
        json.dump(
            value,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        )

    os.replace(temporary, path)


def write_csv(path, dataframe):
    temporary = path.with_name(
        path.name + ".temporary"
    )

    dataframe.to_csv(
        temporary,
        index=False,
        float_format="%.12g"
    )

    os.replace(temporary, path)


# ================================================================
# 3. VERIFY FROZEN NOTEBOOK 02 AND B100 ARTIFACTS
# ================================================================

for name, path in FILES.items():
    need(
        path.is_file(),
        f"Missing frozen artifact: {name}: {path}"
    )

plan = readj(FILES["plan"])
index30 = readj(FILES["index30"])
index100 = readj(FILES["index100"])

audit02 = readj(FILES["audit02"])
protocol100 = readj(FILES["protocol100"])
manifest100 = readj(FILES["manifest100"])

defaults = readj(FILES["defaults"])
software = readj(FILES["software"])

selection = pd.read_csv(FILES["selection"])
blueprints = pd.read_csv(FILES["blueprints"])

PLAN_ID = "3540b29b8201"

need(
    audit02.get("status") == "PASS"
    and audit02.get("plan_id") == PLAN_ID
    and audit02.get("preprocessing") == "B_log",
    "Notebook 02 audit mismatch."
)

need(
    plan.get("experiment_version") == "bootstrap_fixed_grid_v1"
    and plan.get("preprocessing") == "B_log"
    and plan.get("master_seed") == 42
    and plan.get("block_lengths") == [72, 144, 432],
    "Cell 01 frozen plan mismatch."
)

need(
    index30.get("status") == "PASS"
    and index100.get("status") == "PASS"
    and index30["training_sha256"]
    == plan["training_sha256"]
    and index100["source_training_sha256"]
    == plan["training_sha256"],
    "Bootstrap index registration mismatch."
)

need(
    index100["replications"] == list(range(31, 101)),
    "B100 index replication range mismatch."
)

need(
    manifest100.get("status") == "PASS"
    and manifest100.get("total_fits") == 8400
    and manifest100.get("cumulative_B") == 100,
    "B100 completion manifest mismatch."
)

need(
    sha_obj(protocol100) == manifest100["protocol_sha256"],
    "B100 protocol fingerprint mismatch."
)

need(
    sha_file(FILES["metrics100"])
    == manifest100["outputs"]["metrics_sha256"]
    and sha_file(FILES["audit100"])
    == manifest100["outputs"]["audit_sha256"],
    "B100 report checksum mismatch."
)

need(
    sha_file(FILES["metrics30"])
    == manifest100["source_b30_metrics_sha256"],
    "Original B30 metrics changed."
)

for name, key in [
    ("train", "training_sha256"),
    ("validation", "validation_sha256"),
    ("selection", "model_selection_sha256")
]:
    need(
        sha_file(FILES[name]) == plan[key],
        f"Frozen source changed: {name}"
    )


# ================================================================
# 4. VERIFY SOFTWARE VERSIONS
# ================================================================

PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "scikit_learn": "scikit-learn",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm"
}

for label, package in PACKAGES.items():
    current = importlib.metadata.version(package)

    need(
        current == str(software[label]),
        f"Software version mismatch: {package}"
    )


# ================================================================
# 5. FROZEN MODEL AND DATA REGISTRATION
# ================================================================

CONDITIONS = [
    "iid",
    "mbb_L0072",
    "mbb_L0144",
    "mbb_L0432",
    "stationary_L0072",
    "stationary_L0144",
    "stationary_L0432"
]

CLASSES = {
    "Decision Tree": DecisionTreeRegressor,
    "Random Forest": RandomForestRegressor,
    "Bagging": BaggingRegressor,
    "SVR": SVR,
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor
}

SLUGS = {
    "Decision Tree": "decision_tree",
    "Random Forest": "random_forest",
    "Bagging": "bagging",
    "SVR": "svr",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm"
}

ALGOS = list(CLASSES)
SETTINGS = ["default", "tuned"]

KEYS = [
    "condition",
    "replication",
    "algorithm",
    "setting"
]

FEATURES = plan["predictor_order"]

train = pd.read_csv(
    FILES["train"],
    parse_dates=["date"]
)

valid = pd.read_csv(
    FILES["validation"],
    parse_dates=["date"]
)

need(
    len(train) == 11850
    and len(valid) == 4032
    and len(FEATURES) == 27,
    "Frozen dataset dimensions mismatch."
)

need(
    [
        col for col in train.columns
        if col not in ("date", "Appliances")
    ] == FEATURES
    and [
        col for col in valid.columns
        if col not in ("date", "Appliances")
    ] == FEATURES,
    "Frozen feature order changed."
)

need(
    train.date.is_monotonic_increasing
    and valid.date.is_monotonic_increasing
    and train.date.iloc[-1] < valid.date.iloc[0],
    "Temporal boundary mismatch."
)

Xtrain = train[FEATURES]
ytrain = train["Appliances"]

Xval = valid[FEATURES]
yval = valid["Appliances"].to_numpy(dtype=float)

N = len(train)

need(
    np.isfinite(Xtrain.to_numpy(dtype=float)).all()
    and np.isfinite(Xval.to_numpy(dtype=float)).all()
    and np.isfinite(ytrain.to_numpy(dtype=float)).all()
    and np.isfinite(yval).all()
    and ytrain.ge(0).all()
    and (yval >= 0).all(),
    "Nonfinite or negative frozen data values."
)

need(
    set(defaults) == set(ALGOS)
    and len(selection) == 6
    and selection.algorithm.is_unique
    and set(selection.algorithm) == set(ALGOS),
    "Model registration mismatch."
)

CANDIDATE = dict(
    zip(
        selection.algorithm,
        selection.candidate_id.astype(str)
    )
)

TUNED = {
    str(row.algorithm): json.loads(row.parameters_json)
    for row in selection.itertuples(index=False)
}

need(
    plan["model_selection"] == CANDIDATE,
    "Tuned candidate IDs changed."
)

blue = blueprints[
    blueprints.configuration == "B_log"
]

need(
    len(blue) == 6
    and set(blue.algorithm) == set(ALGOS)
    and blue.n_features.eq(27).all()
    and blue.target_treatment.eq("log1p").all(),
    "B_log blueprint mismatch."
)

for row in blue.itertuples(index=False):
    scaler = (
        "none" if pd.isna(row.scaler)
        else str(row.scaler).lower()
    )

    expected_scaler = (
        "standardscaler"
        if row.algorithm == "SVR"
        else "none"
    )

    need(
        scaler == expected_scaler
        and str(row.example_parameter_key).startswith(
            "regressor__model__"
        ),
        f"Pipeline blueprint mismatch: {row.algorithm}"
    )


def model_factory(algo, setting):
    estimator = CLASSES[algo](
        **dict(defaults[algo])
    )

    if setting == "tuned":
        if algo == "Bagging":
            estimator.set_params(
                estimator=DecisionTreeRegressor(
                    random_state=42
                )
            )

        estimator.set_params(
            **TUNED[algo]
        )

    steps = (
        [("scaler", StandardScaler())]
        if algo == "SVR"
        else []
    )

    steps.append(("model", estimator))

    return TransformedTargetRegressor(
        regressor=Pipeline(steps),
        func=np.log1p,
        inverse_func=np.expm1
    )


def score(prediction):
    prediction = np.asarray(
        prediction,
        dtype=np.float64
    ).reshape(-1)

    need(
        prediction.shape == (len(yval),)
        and np.isfinite(prediction).all(),
        "Invalid validation prediction."
    )

    return {
        "rmse_wh": float(
            np.sqrt(
                mean_squared_error(yval, prediction)
            )
        ),
        "mae_wh": float(
            mean_absolute_error(yval, prediction)
        ),
        "r2": float(
            r2_score(yval, prediction)
        )
    }


# ================================================================
# 6. VERIFY ALL B100 METRIC RECORDS
# ================================================================

b100 = pd.read_csv(FILES["metrics100"])
audit100 = pd.read_csv(FILES["audit100"])

need(
    len(b100) == 8400
    and not b100.duplicated(KEYS).any()
    and b100.status.eq("PASS").all()
    and set(b100.condition) == set(CONDITIONS),
    "B100 result table invalid."
)

need(
    b100.groupby(
        ["condition", "replication"]
    ).size().eq(12).all()
    and b100.groupby(
        ["condition", "replication"]
    ).index_sha256.nunique().eq(1).all(),
    "B100 model pairing mismatch."
)

need(
    b100.groupby("condition")
    .replication.nunique().eq(100).all()
    and b100.replication.between(1, 100).all(),
    "B100 replication coverage mismatch."
)

need(
    len(audit100) == 700
    and audit100.model_fits.eq(12).all()
    and audit100.index_fingerprints.eq(1).all(),
    "B100 replication audit mismatch."
)

need(
    np.isfinite(
        b100[
            ["rmse_wh", "mae_wh", "r2", "fit_seconds"]
        ].to_numpy(dtype=float)
    ).all(),
    "Nonfinite B100 metrics."
)

for row in b100.itertuples(index=False):
    expected = (
        "default"
        if row.setting == "default"
        else CANDIDATE[row.algorithm]
    )

    need(
        row.candidate_id == expected,
        "B100 model candidate mismatch."
    )


# ================================================================
# 7. ORIGINAL CELL 01 BOOTSTRAP GENERATORS
# ================================================================

def unpack(condition):
    if condition == "iid":
        return "iid", None

    method, label = condition.rsplit("_L", 1)
    return method, int(label)


def seed(method, ell, replication):
    key = (
        f"{plan['experiment_version']}|"
        f"{PLAN_ID}|"
        f"{plan['training_sha256']}|"
        f"{plan['master_seed']}|"
        f"{method}|{ell}|{replication}"
    )

    return int.from_bytes(
        hashlib.sha256(
            key.encode("utf-8")
        ).digest()[:16],
        "little"
    )


def draw(method, ell, replication):
    rng = np.random.default_rng(
        seed(method, ell, replication)
    )

    if method == "iid":
        return rng.integers(
            low=0,
            high=N,
            size=N,
            dtype=np.int32
        )

    output = np.empty(N, dtype=np.int32)

    if method == "mbb":
        starts = rng.integers(
            low=0,
            high=N - ell + 1,
            size=math.ceil(N / ell)
        )

        position = 0

        for start in starts:
            used = min(ell, N - position)

            output[
                position:position + used
            ] = np.arange(
                int(start),
                int(start) + used
            )

            position += used

    elif method == "stationary":
        position = 0
        p = 1.0 / ell

        while position < N:
            raw_length = int(
                rng.geometric(p)
            )

            start = int(
                rng.integers(0, N)
            )

            used = min(
                raw_length,
                N - position
            )

            output[
                position:position + used
            ] = (
                start + np.arange(used)
            ) % N

            position += used

    else:
        raise ValueError(
            f"Unknown bootstrap method: {method}"
        )

    return output


# ================================================================
# 8. VERIFY OLD INDICES, GENERATE NEW 101–300
# ================================================================

new_index = {}
index_registration = {}

for condition in CONDITIONS:
    method, ell = unpack(condition)

    old30 = (
        IDX30
        / index30["checkpoints"][condition]["file"]
    )

    old100 = (
        B100 / "resampling_indices_extension"
        / index100["checkpoints"][condition]["file"]
    )

    need(
        old30.is_file()
        and sha_file(old30)
        == index30["checkpoints"][condition]["sha256"],
        f"B30 indices changed: {condition}"
    )

    need(
        old100.is_file()
        and sha_file(old100)
        == index100["checkpoints"][condition]["sha256"],
        f"B100 indices changed: {condition}"
    )

    with np.load(old30, allow_pickle=False) as stored:
        matrix30 = stored["indices"].copy()

    with np.load(old100, allow_pickle=False) as stored:
        matrix100 = stored["indices"].copy()

    need(
        matrix30.shape == (30, N)
        and matrix100.shape == (70, N),
        "Historical index matrix mismatch."
    )

    # Verify ALL old index fingerprints against B100 CSV.
    recorded = (
        b100.loc[b100.condition == condition]
        .groupby("replication")
        .index_sha256.first()
    )

    for rep in range(1, 101):
        old = (
            matrix30[rep - 1]
            if rep <= 30
            else matrix100[rep - 31]
        )

        need(
            sha_array(old) == recorded.loc[rep],
            f"Historical index mismatch: {condition}, {rep}"
        )

    # Seed-rule replay at historical boundaries.
    for rep in (1, 30, 31, 100):
        old = (
            matrix30[rep - 1]
            if rep <= 30
            else matrix100[rep - 31]
        )

        need(
            np.array_equal(
                draw(method, ell, rep),
                old
            ),
            f"Seed replay failed: {condition}, {rep}"
        )

    # Only new replications are generated.
    matrix = np.stack(
        [
            draw(method, ell, rep)
            for rep in range(101, 301)
        ],
        axis=0
    )

    need(
        matrix.shape == (200, N)
        and matrix.min() >= 0
        and matrix.max() < N,
        f"Invalid new indices: {condition}"
    )

    destination = (
        IDX300
        / f"{condition}_B0101_0300.npz"
    )

    if destination.exists():
        with np.load(
            destination,
            allow_pickle=False
        ) as stored:
            need(
                set(stored.files) == {"indices"}
                and np.array_equal(
                    stored["indices"],
                    matrix
                ),
                f"Existing B300 indices differ: {condition}"
            )
    else:
        temporary = destination.with_name(
            destination.name + ".temporary"
        )

        with temporary.open("wb") as file:
            np.savez_compressed(
                file,
                indices=matrix
            )

        os.replace(temporary, destination)

    new_index[condition] = matrix

    index_registration[condition] = {
        "file": destination.name,
        "sha256": sha_file(destination)
    }


INDEX_REG = (
    IDX300 / "B0101_0300_checkpoint_registry.json"
)

registry = {
    "status": "PASS",
    "experiment_version": plan["experiment_version"],
    "source_training_sha256": plan["training_sha256"],
    "replications": [101, 300],
    "checkpoints": index_registration
}

if INDEX_REG.exists():
    need(
        readj(INDEX_REG) == registry,
        "B300 index registry differs."
    )
else:
    write_json(INDEX_REG, registry)


# ================================================================
# 9. FREEZE B300 CONTINUATION PROTOCOL
# ================================================================

PROTOCOL = {
    "cell_version": "cell02d_cumulative_B300_v1",
    "plan_id": PLAN_ID,

    "source_B100_protocol_sha256": sha_obj(protocol100),
    "source_B100_metrics_sha256": sha_file(
        FILES["metrics100"]
    ),
    "source_B100_audit_sha256": sha_file(
        FILES["audit100"]
    ),
    "source_plan_sha256": sha_file(FILES["plan"]),

    "source_training_sha256": plan["training_sha256"],
    "source_validation_sha256": plan["validation_sha256"],
    "source_selection_sha256": plan[
        "model_selection_sha256"
    ],

    "defaults_sha256": sha_file(FILES["defaults"]),
    "blueprints_sha256": sha_file(FILES["blueprints"]),
    "new_index_registry_sha256": sha_file(INDEX_REG),

    "conditions": CONDITIONS,
    "algorithm_order": ALGOS,
    "settings": SETTINGS,
    "model_seed": 42,

    "additional_replications": [101, 300],
    "cumulative_B": 300,

    "transform": "log1p/expm1",
    "prediction_clipping": False,
    "fixed_validation_size": len(yval),
    "test_accessed": False
}

PROTOCOL_HASH = sha_obj(PROTOCOL)

PROTOCOL_FILE = (
    REPORT300 / "cell02d_B300_extension_protocol.json"
)

if PROTOCOL_FILE.exists():
    need(
        readj(PROTOCOL_FILE) == PROTOCOL,
        "Existing B300 continuation protocol differs."
    )
else:
    write_json(PROTOCOL_FILE, PROTOCOL)


# ================================================================
# 10. RESUMABLE MODEL CHECKPOINT FUNCTIONS
# ================================================================

def identity(condition, rep, algo, setting, index_hash):
    return {
        "cell_version": PROTOCOL["cell_version"],
        "protocol_sha256": PROTOCOL_HASH,
        "condition": condition,
        "replication": rep,
        "algorithm": algo,
        "setting": setting,
        "candidate_id": (
            "default"
            if setting == "default"
            else CANDIDATE[algo]
        ),
        "index_sha256": index_hash
    }


def verify_ckpt(path, expected):
    with np.load(
        path,
        allow_pickle=False
    ) as stored:
        need(
            set(stored.files)
            == {"prediction_wh", "metadata"},
            f"Invalid checkpoint schema: {path}"
        )

        prediction = stored["prediction_wh"].copy()

        metadata = json.loads(
            stored["metadata"].item()
        )

    for key, value in expected.items():
        need(
            metadata.get(key) == value,
            f"Checkpoint identity mismatch: {path}, {key}"
        )

    need(
        sha_array(prediction)
        == metadata["prediction_sha256"],
        f"Prediction hash mismatch: {path}"
    )

    for key, value in score(prediction).items():
        need(
            np.isclose(
                value,
                metadata[key],
                rtol=1e-10,
                atol=1e-8
            ),
            f"Checkpoint metric mismatch: {path}, {key}"
        )

    return metadata


def save_ckpt(path, prediction, metadata):
    need(
        not path.exists(),
        "Refusing checkpoint overwrite."
    )

    temporary = path.with_name(
        path.name + ".temporary"
    )

    if temporary.exists():
        temporary.unlink()

    with temporary.open("wb") as file:
        np.savez_compressed(
            file,
            prediction_wh=np.asarray(
                prediction,
                dtype=np.float64
            ),
            metadata=np.array(
                canonical(metadata)
            )
        )

    os.replace(temporary, path)


# ================================================================
# 11. EXECUTE REPLICATIONS 101–300
# ================================================================

print("=" * 72)
print("CELL 02D — B=300 CUMULATIVE BOOTSTRAP REFITTING")
print("=" * 72)

print("Preserved B100 fits:", len(b100))
print("Additional planned fits:", 16800)
print("Cumulative target:", 25200)
print("Historical indices: VERIFIED")
print("New index conditions:", len(new_index))
print("Protocol SHA256:", PROTOCOL_HASH)

records = []
fresh = 0
resumed = 0

for condition in CONDITIONS:
    print(
        "\nCONDITION:",
        condition,
        flush=True
    )

    matrix = new_index[condition]

    for rep in range(101, 301):
        indices = matrix[rep - 101]
        index_hash = sha_array(indices)

        # Same X-y row pairs and same bootstrap
        # indices for all 12 model configurations.
        X_boot = (
            Xtrain.iloc[indices]
            .reset_index(drop=True)
        )

        y_boot = (
            ytrain.iloc[indices]
            .reset_index(drop=True)
        )

        directory = (
            CHECK300 / condition / f"rep_{rep:03d}"
        )

        directory.mkdir(
            parents=True,
            exist_ok=True
        )

        rep_new = 0
        rep_resumed = 0

        for algo in ALGOS:
            for setting in SETTINGS:

                path = (
                    directory
                    / f"{SLUGS[algo]}__{setting}.npz"
                )

                expected = identity(
                    condition,
                    rep,
                    algo,
                    setting,
                    index_hash
                )

                if path.exists():
                    metadata = verify_ckpt(
                        path,
                        expected
                    )

                    rep_resumed += 1
                    resumed += 1

                else:
                    model = model_factory(
                        algo,
                        setting
                    )

                    started = time.perf_counter()

                    try:
                        # Suppress repetitive LightGBM
                        # informational stdout only.
                        # Estimator parameters are unchanged.
                        if algo == "LightGBM":
                            with contextlib.redirect_stdout(
                                io.StringIO()
                            ):
                                model.fit(X_boot, y_boot)
                        else:
                            model.fit(X_boot, y_boot)

                        fit_seconds = (
                            time.perf_counter() - started
                        )

                        prediction = np.asarray(
                            model.predict(Xval),
                            dtype=np.float64
                        ).reshape(-1)

                        metrics = score(prediction)

                    except Exception as error:
                        raise RuntimeError(
                            f"Fit failed: {condition}, "
                            f"rep {rep}, {algo}, {setting}. "
                            "Existing checkpoints remain intact."
                        ) from error

                    metadata = {
                        **expected,
                        **metrics,
                        "fit_seconds": float(fit_seconds),
                        "sample_size": N,
                        "validation_size": len(yval),
                        "model_seed": 42,
                        "target_scale": "Wh",
                        "preprocessing": "B_log",
                        "prediction_sha256": sha_array(
                            prediction
                        ),
                        "status": "PASS"
                    }

                    save_ckpt(
                        path,
                        prediction,
                        metadata
                    )

                    metadata = verify_ckpt(
                        path,
                        expected
                    )

                    del model

                    rep_new += 1
                    fresh += 1

                records.append({
                    "condition": condition,
                    "replication": rep,
                    "algorithm": algo,
                    "setting": setting,
                    "candidate_id": metadata["candidate_id"],
                    "rmse_wh": metadata["rmse_wh"],
                    "mae_wh": metadata["mae_wh"],
                    "r2": metadata["r2"],
                    "fit_seconds": metadata["fit_seconds"],
                    "index_sha256": index_hash,
                    "prediction_sha256": metadata[
                        "prediction_sha256"
                    ],
                    "checkpoint_path": str(path),
                    "status": "PASS"
                })

        need(
            rep_new + rep_resumed == 12,
            f"Incomplete replication: {condition}, {rep}"
        )

        print(
            f"{condition} | rep {rep:03d}/300 | "
            f"new={rep_new} | resumed={rep_resumed} | PASS",
            flush=True
        )


# ================================================================
# 12. COMBINE ORIGINAL B100 AND NEW B101–300
# ================================================================

extension = pd.DataFrame.from_records(
    records,
    columns=b100.columns
)

need(
    len(extension) == 16800
    and not extension.duplicated(KEYS).any()
    and extension.replication.between(101, 300).all(),
    "B300 extension incomplete."
)

combined = pd.concat(
    [b100, extension],
    ignore_index=True
)

need(
    len(combined) == 25200
    and not combined.duplicated(KEYS).any()
    and combined.status.eq("PASS").all()
    and combined.groupby("condition").size().eq(3600).all(),
    "Cumulative B300 records incomplete."
)

need(
    np.isfinite(
        combined[
            ["rmse_wh", "mae_wh", "r2", "fit_seconds"]
        ].to_numpy(dtype=float)
    ).all(),
    "Nonfinite cumulative metrics."
)

groups = (
    combined.groupby(
        ["condition", "replication"],
        sort=False
    )
    .agg(
        model_fits=("algorithm", "size"),
        algorithms=("algorithm", "nunique"),
        settings=("setting", "nunique"),
        index_fingerprints=(
            "index_sha256",
            "nunique"
        )
    )
    .reset_index()
)

need(
    len(groups) == 2100
    and groups.model_fits.eq(12).all()
    and groups.algorithms.eq(6).all()
    and groups.settings.eq(2).all()
    and groups.index_fingerprints.eq(1).all(),
    "Cumulative pairing audit failed."
)

need(
    groups.groupby("condition")
    .replication.nunique().eq(300).all(),
    "B300 replication coverage incomplete."
)


# ================================================================
# 13. STABLE ORDER AND SAVE FINAL B300 OUTPUTS
# ================================================================

combined["_condition_sort"] = pd.Categorical(
    combined.condition,
    categories=CONDITIONS,
    ordered=True
)

combined["_algorithm_sort"] = pd.Categorical(
    combined.algorithm,
    categories=ALGOS,
    ordered=True
)

combined["_setting_sort"] = pd.Categorical(
    combined.setting,
    categories=SETTINGS,
    ordered=True
)

combined = (
    combined.sort_values(
        [
            "_condition_sort",
            "replication",
            "_algorithm_sort",
            "_setting_sort"
        ],
        kind="stable"
    )
    .drop(
        columns=[
            "_condition_sort",
            "_algorithm_sort",
            "_setting_sort"
        ]
    )
    .reset_index(drop=True)
)

METRICS_PATH = (
    REPORT300
    / "bootstrap_B300_all_replication_metrics.csv"
)

AUDIT_PATH = (
    REPORT300
    / "bootstrap_B300_replication_audit.csv"
)

MANIFEST_PATH = (
    REPORT300
    / "bootstrap_B300_completion_manifest.json"
)

write_csv(METRICS_PATH, combined)
write_csv(AUDIT_PATH, groups)

manifest = {
    "cell": "02D",
    "status": "PASS",
    "cumulative_B": 300,
    "original_B100_fits_reused": 8400,
    "extension_101_300_fits": 16800,
    "fresh_fits_this_run": fresh,
    "resumed_extension_fits": resumed,
    "total_fits": 25200,
    "protocol_sha256": PROTOCOL_HASH,
    "conditions": CONDITIONS,
    "validation_rows": len(yval),
    "hyperparameters_retuned": False,
    "test_accessed": False,
    "historical_1_100_refitted": False,
    "outputs": {
        "metrics_csv": str(METRICS_PATH),
        "metrics_sha256": sha_file(METRICS_PATH),
        "audit_csv": str(AUDIT_PATH),
        "audit_sha256": sha_file(AUDIT_PATH)
    }
}

write_json(MANIFEST_PATH, manifest)


# ================================================================
# 14. FINAL REPORT
# ================================================================

print("\n" + "=" * 72)
print("CELL 02D — CUMULATIVE B300 COMPLETED")
print("=" * 72)

print("B100 fits reused:", 8400)
print("New fitting operations:", fresh)
print("Resumed B101–300 fits:", resumed)
print("Cumulative records:", len(combined))

condition_audit = (
    groups.groupby("condition", sort=False)
    .agg(
        repetitions=("replication", "nunique"),
        model_fits=("model_fits", "sum"),
        shared_indices=(
            "index_fingerprints",
            lambda values: bool(
                (values == 1).all()
            )
        )
    )
    .reset_index()
)

display(condition_audit)

print("\nOUTPUTS")
print("Metrics:", METRICS_PATH)
print("Audit:", AUDIT_PATH)
print("Manifest:", MANIFEST_PATH)

print("\nFINAL STATUS: PASS")
print("Historical B1–100 refitted: NO")
print("Hyperparameters retuned: NO")
print("Test data accessed: NO")
print("Figures generated: NO")


CELL 02D — B=300 CUMULATIVE BOOTSTRAP REFITTING
Preserved B100 fits: 8400
Additional planned fits: 16800
Cumulative target: 25200
Historical indices: VERIFIED
New index conditions: 7
Protocol SHA256: adae65150016c6357d874ccd4634d8d59cb714eb29ee010ddbb7de6ad3a7403f

CONDITION: iid
iid | rep 101/300 | new=0 | resumed=12 | PASS
iid | rep 102/300 | new=0 | resumed=12 | PASS
iid | rep 103/300 | new=0 | resumed=12 | PASS
iid | rep 104/300 | new=0 | resumed=12 | PASS
iid | rep 105/300 | new=0 | resumed=12 | PASS
iid | rep 106/300 | new=0 | resumed=12 | PASS
iid | rep 107/300 | new=0 | resumed=12 | PASS
iid | rep 108/300 | new=0 | resumed=12 | PASS
iid | rep 109/300 | new=0 | resumed=12 | PASS
iid | rep 110/300 | new=0 | resumed=12 | PASS
iid | rep 111/300 | new=0 | resumed=12 | PASS
iid | rep 112/300 | new=0 | resumed=12 | PASS
iid | rep 113/300 | new=0 | resumed=12 | PASS
iid | rep 114/300 | new=0 | resumed=12 | PASS
iid | rep 115/300 | new=0 | resumed=12 | PASS
iid | rep 116/300 | new=0 | r

,condition,repetitions,model_fits,shared_indices
0,iid,300,3600,True
1,mbb_L0072,300,3600,True
2,mbb_L0144,300,3600,True
3,mbb_L0432,300,3600,True
4,stationary_L0072,300,3600,True
5,stationary_L0144,300,3600,True
6,stationary_L0432,300,3600,True



OUTPUTS
Metrics: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B300/reports/bootstrap_B300_all_replication_metrics.csv
Audit: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B300/reports/bootstrap_B300_replication_audit.csv
Manifest: /content/drive/MyDrive/Stability_Driven_Energy/04_bootstrap/model_refitting_B300/reports/bootstrap_B300_completion_manifest.json

FINAL STATUS: PASS
Historical B1–100 refitted: NO
Hyperparameters retuned: NO
Test data accessed: NO
Figures generated: NO
